# October 14, 2026

## Prep

In [1]:
import pandas as pd
import numpy as np 
from tabulate import tabulate
from simnibs_parcel_analysis import summarize_treatment_courses

fn_cgi   = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_latest.xlsx'
fn_query = '/projects/p32903/Alex2/ect-results-tools/query_result_1.csv'

def _add_person_id(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["person_id"] = df["subjid"].str.extract(
        r"^(subj-cat-\d{3})(?:-\d{3})?$", expand=False
    )

    invalid = df["person_id"].isna()
    if invalid.any():
        raise ValueError(
            f"Invalid/missing subjid at rows {df.index[invalid].tolist()}"
        )
    return df

def load_stimulation_data(fn: str) -> pd.DataFrame:
    df = pd.read_excel(fn, sheet_name="stimulus")
    return _add_person_id(df)

def load_course_data(fn: str) -> pd.DataFrame:
    df = pd.read_excel(fn, sheet_name="Sheet1")
    df = df.loc[
        df["in_stim"].eq(True) & df["batch_num"].gt(0)
    ].copy()

    df["cgi_change"] = df["cgi_start"] - df["cgi_end"]
    return _add_person_id(df)

def load_query_results(fn: str) -> pd.DataFrame:
    fs_subjects_dirs = {
        "STU00225089": "/projects/p32903/Alex2/datasets/STU00225089/subjectsdir",
        "STU00225089_synthsr": "/projects/p32903/Alex2/datasets/STU00225089/synthsr/subjectsdir"
    }
    df_query = pd.read_csv(fn)
    df_query["fs_subjects_dir"] = df_query["dataset_root"].map(fs_subjects_dirs)

    if df_query["fs_subjects_dir"].isna().any():
        missing = df_query.loc[df_query["fs_subjects_dir"].isna(), "dataset_root"].unique()
        raise ValueError(f"Missing FreeSurfer SUBJECTS_DIR mapping for: {missing}")
    return df_query

def _compute_ef_sf(df:pd.DataFrame)->pd.DataFrame:
    cols    = ['pulse_width_ms', 'frequency_hz', 'percent_charge']
    df      = df.copy()
    df['ef_sf'] = df['pulse_width_ms'] * df['frequency_hz'] * df['percent_charge']
    return df


## 1) LMM

### 1.1) `Tx ~ 1 + age*sex + electrode_placement`


Using second session parameters to summarize treatment course

In [ ]:
def run_null(df:pd.DataFrame):
    import statsmodels.formula.api as smf
    df = _prepare_df_null()
    formula = "cgi_change ~ 1 + age * C(sex) + C(electrode_placement)"
    return smf.ols(formula, data=df, missing="raise").fit()

### COLLAPSE SESSION -> COURSE BY SECOND SESSION ###
def _prepare_df_null()->pd.DataFrame:
    fn_cgi   = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_latest.xlsx'

    df_cgi  = load_course_data(fn_cgi)
    df_stim = load_stimulation_data(fn_cgi)
    df_stim = df_stim[df_stim['subjid'] != 'subj-cat-010-001']
    
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy() # only keep subjs present in both tables
    
    df_stim = df_stim[df_stim['session_num']==2]
    df_stim.drop(columns=['session_num'])

    # cgi_cols_keep   = ['subjid', 'sex', 'age', 'cgi_start','series_num', 'cgi_end', 'cgi_change', 'response_type']
    # df_cgi_filtered = df_cgi[cgi_cols_keep].copy()

    shared_metadata = [
    col for col in ["age", "sex", "dob"]
        if col in df_stim.columns and col in df_cgi.columns
    ]

    df_combined = df_stim.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
        )
    return df_combined

df = _prepare_df_null()
print(f"Number of treatment courses: {df.shape[0]}")
print(f"Number of unique patients: {df.person_id.unique().shape[0]}")
result = run_null(df)
result.summary()

### 1.2) `Tx ~ 1 + E_x + E_x:ef_sf + (1|age) + (1|sex)`

In [ ]:
"""ROI-wise ECT mixed models using the supplied HCP-MMP1 fsaverage workflow.

In Jupyter, define load_query_results, load_course_data, load_stimulation_data,
and _compute_ef_sf first, then run: %run -i roi_lmm.py

No analysis runs merely by loading this file. Meshes are read only in preparation.
The exact fixed-effects model is cgi_change ~ 1 + E_x + E_x:ef_sf.
Age and sex remain crossed categorical random intercepts, as requested.
"""
from __future__ import annotations

from numbers import Real

from collections.abc import Sequence
from pathlib import Path
from typing import Literal
import time
import warnings

import numpy as np
import pandas as pd
import statsmodels
import statsmodels.formula.api as smf
from tqdm import tqdm


def _prepare_df_lmm_roi(
    fn_query: str | Path,
    fn_cgi: str | Path,
    *,
    threshold: int = 95,
    step: str = "step_4",
    model_type: str = "skin_double",
    asf: str = "asf_5",
    atlas_name: str = "HCP_MMP1",
    percentile: float = 95,
    percentile_weighting: Literal["node", "area"] = "node",
    combine_hemis: bool = False,
    representative_session: int = 2,
    exclude_rois: Sequence[str] = (),
    subject_scan_map: str | Path = "subject_scan_map.json",) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return one row/course with ROI fields, plus one metadata row/ROI metric.

    Uses the four existing notebook loaders/helpers named in the module docstring.
    Clinical age/sex/dob take precedence over session metadata when both exist.
    No global E, normalization, dose multiplication, or modeling occurs here.
    Bilateral fields are equal-weight means of the two hemisphere summaries.
    """
    from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
    from simnibs_parcel_analysis.fsaverage_roi_metrics import extract_fsaverage_roi_metrics

    if (isinstance(percentile, (bool, np.bool_)) or not isinstance(percentile, Real)
            or not np.isfinite(percentile) or not 0 <= percentile <= 100):
        raise ValueError("percentile must be finite and in [0, 100].")
    if percentile_weighting not in ("node", "area"):
        raise ValueError("Surface percentile_weighting must be 'node' or 'area'.")
    if not isinstance(combine_hemis, (bool, np.bool_)):
        raise TypeError("combine_hemis must be a boolean.")
    if (isinstance(representative_session, bool) or not isinstance(representative_session, int)
            or representative_session < 1):
        raise ValueError("representative_session must be a positive integer.")
    if isinstance(exclude_rois, str):
        raise TypeError("exclude_rois must be a sequence, not one string.")

    excluded_subject = "subj-cat-010-001"
    query = load_query_results(fn_query)
    query = query.loc[query["subject_dir"].ne(excluded_subject)].copy()
    cgi = load_course_data(fn_cgi)
    stim = _compute_ef_sf(load_stimulation_data(fn_cgi))
    stim = stim.loc[stim["subjid"].ne(excluded_subject) & stim["person_id"].isin(cgi["person_id"])].copy()

    filters = {"model_type_dir": model_type, "step_result": "fsavg_overlays", "step": step,
               "asf": asf, "threshold": threshold}
    mapped = map_session_files(
        sessions=stim, query=query, filename="tdcs_uq_gpc_fsavg.msh",
        query_filters=filters, subject_scan_map=str(subject_scan_map),
    )
    mapped = mapped.loc[mapped["session_num"].eq(representative_session)].copy()
    if mapped.empty:
        raise ValueError(f"No eligible session {representative_session} rows for {filters}")

    keys = ["person_id", "series_num"]
    for label, frame in (("Mapped sessions", mapped), ("CGI table", cgi)):
        _require_columns(frame, keys, label)
        if frame[keys].isna().any().any() or frame.duplicated(keys).any():
            raise ValueError(f"{label}: missing or duplicated person_id/series_num.")
    _require_columns(mapped, ["full_file_path"], "Mapped sessions")
    if mapped["full_file_path"].isna().any():
        raise ValueError("Missing mapped mesh paths.")
    if "full_file_path" in cgi.columns or "_course_match" in mapped or "_course_match" in cgi:
        raise ValueError("Clinical merge would conflict with a reserved column.")

    shared = [c for c in ("age", "sex", "dob") if c in mapped and c in cgi]
    courses = mapped.drop(columns=shared).merge(
        cgi, on=keys, how="left", sort=False, validate="one_to_one",
        suffixes=("_stim", "_cgi"), indicator="_course_match",
    )
    if courses["_course_match"].ne("both").any():
        bad = courses.loc[courses["_course_match"].ne("both"), keys].to_dict("records")
        raise ValueError(f"Selected sessions have no matching CGI course: {bad}")
    courses = courses.drop(columns="_course_match")
    _require_columns(courses, ["cgi_change", "ef_sf", "age", "sex"], "Courses")
    if courses[["cgi_change", "ef_sf", "age", "sex"]].isna().any().any():
        raise ValueError("Missing model variables in the selected courses; no rows were dropped.")

    # Each unique mesh is extracted once, even if multiple courses reuse it.
    paths = courses[["full_file_path"]].drop_duplicates()
    extracted = extract_fsaverage_roi_metrics(
        paths, path_column="full_file_path", field_name="magnE_mean", atlas_name=atlas_name,
        min_nodes_per_roi=20, exclude_rois=tuple(exclude_rois), percentile=float(percentile),
        percentile_weighting=percentile_weighting, output_format="wide",
    )
    _require_columns(extracted, ["full_file_path"], "ROI extraction")
    roi_cols = [c for c in extracted if isinstance(c, str) and c.startswith(("lh.", "rh."))
                and c.endswith(("_e_mean", "_e_percentile"))]
    if not roi_cols:
        raise ValueError("No hemisphere-specific mean/percentile ROI columns were extracted.")
    roi = extracted[["full_file_path", *roi_cols]].copy()
    if roi["full_file_path"].duplicated().any() or set(roi["full_file_path"]) != set(paths["full_file_path"]):
        raise ValueError("ROI extraction must return exactly one row for every unique mesh.")

    if combine_hemis:
        lh = {c[3:]: c for c in roi_cols if c.startswith("lh.")}
        rh = {c[3:]: c for c in roi_cols if c.startswith("rh.")}
        if not lh or set(lh) != set(rh):
            raise ValueError(f"Unmatched hemisphere ROI metrics: {sorted(set(lh) ^ set(rh))}")
        bilateral = pd.DataFrame({r: (roi[lh[r]] + roi[rh[r]]) / 2 for r in sorted(lh)}, index=roi.index)
        roi = pd.concat([roi[["full_file_path"]], bilateral], axis=1)
        roi_cols = list(bilateral.columns)

    overlap = set(roi_cols) & set(courses.columns)
    if overlap:
        raise ValueError(f"ROI columns conflict with clinical/session columns: {sorted(overlap)}")
    prepared = courses.merge(roi, on="full_file_path", how="left", sort=False, validate="many_to_one")

    config = {
        "atlas": atlas_name, "domain": "fsaverage_surface", "field_name": "magnE_mean",
        "model_type": model_type, "step": step, "threshold": threshold, "asf": asf,
        "combine_hemis": bool(combine_hemis), "representative_session": representative_session,
        "hemisphere_aggregation": "mean_of_hemisphere_summaries" if combine_hemis else "none",
        "excluded_subject": excluded_subject, "excluded_rois": "|".join(exclude_rois),
        "query_file": str(fn_query), "clinical_file": str(fn_cgi),
    }
    metadata = []
    for col in roi_cols:
        metric = "mean" if col.endswith("_e_mean") else "percentile"
        label = col.removesuffix(f"_e_{metric}")
        hemi, name = ("bilateral", label) if combine_hemis else tuple(label.split(".", 1))
        metadata.append({
            **config, "roi_column": col, "roi": name, "hemisphere": hemi, "metric": metric,
            "percentile": float(percentile) if metric == "percentile" else np.nan,
            "mean_weighting": "area" if metric == "mean" else None,
            "percentile_weighting": percentile_weighting if metric == "percentile" else None,
        })
    return prepared, pd.DataFrame(metadata)


def _require_columns(df: pd.DataFrame, columns: Sequence[str], label: str) -> None:
    if not df.columns.is_unique:
        raise ValueError(f"{label}: duplicate column names.")
    missing = set(columns) - set(df.columns)
    if missing:
        raise ValueError(f"{label}: missing columns {sorted(missing)}")


def run_roi_lmm(
    df: pd.DataFrame,
    roi_info: pd.DataFrame,
    output_csv: str | Path,
    *,
    metrics: Sequence[Literal["mean", "percentile"]] = ("percentile",),
    reml: bool = True,
    optimizer: str = "lbfgs",
    maxiter: int = 1000,
    on_error: Literal["raise", "record"] = "raise",
    overwrite: bool = False,) -> pd.DataFrame:
    """Fit one mixed-effects model per ROI/metric and save results incrementally.

    Fixed effects: cgi_change ~ 1 + E_x + E_x:ef_sf
    Random effects: crossed categorical intercepts for age and sex.

    No rows are dropped, and predictors are not centered or standardized.
    """
    if (isinstance(metrics, str) or not metrics or len(set(metrics)) != len(metrics)
            or not set(metrics).issubset({"mean", "percentile"})):
        raise ValueError("metrics must contain unique choices from 'mean' and 'percentile'.")
    if on_error not in ("raise", "record"):
        raise ValueError("on_error must be 'raise' or 'record'.")
    if not isinstance(reml, bool) or isinstance(maxiter, bool) or not isinstance(maxiter, int) or maxiter < 1:
        raise ValueError("reml must be boolean and maxiter a positive integer.")

    # Select the requested ROI summary statistics.
    _require_columns(roi_info, ["roi_column", "roi", "hemisphere", "metric"], "ROI metadata")
    spec = roi_info.loc[roi_info["metric"].isin(metrics)].copy()

    if spec.empty or not set(metrics).issubset(set(spec["metric"])) or spec["roi_column"].duplicated().any():
        raise ValueError("Requested ROI metrics are absent or repeated in ROI metadata.")
    if spec[["roi_column", "roi", "hemisphere", "metric"]].isna().any().any():
        raise ValueError("ROI metadata contains missing identifiers.")

    # Strict input validation before fitting any models.
    keys = ["person_id", "series_num"]
    common = [*keys, "cgi_change", "ef_sf", "age", "sex"]
    roi_cols = spec["roi_column"].tolist()

    _require_columns(df, [*common, *roi_cols], "Prepared data")
    if df.empty or df[[*common, *roi_cols]].isna().any().any():
        raise ValueError("Empty data or missing model inputs; no rows were dropped.")
    if df.duplicated(keys).any():
        raise ValueError("Duplicate treatment courses: person_id/series_num.")

    data = df[[*common, *roi_cols]].copy().reset_index(drop=True)
    numeric = ["cgi_change", "ef_sf", "age", *roi_cols]
    data[numeric] = data[numeric].apply(pd.to_numeric, errors="raise")

    if not np.isfinite(data[numeric].to_numpy(dtype=float)).all():
        raise ValueError("Non-finite model inputs.")
    if (data[roi_cols] < 0).any().any() or data["ef_sf"].le(0).any():
        raise ValueError("ROI magnitudes must be nonnegative and ef_sf positive.")
    if not data["sex"].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError("sex must contain nonblank string labels; normalize labels explicitly beforehand.")

    data["sex"] = data["sex"].astype("category").cat.remove_unused_categories()
    n, n_age, n_sex = len(data), data["age"].nunique(), data["sex"].nunique()

    if n <= 3 or data["cgi_change"].nunique() < 2:
        raise ValueError("Need more than three courses and a nonconstant outcome.")
    if n_age == n:
        raise ValueError("Every age is unique: (1|age) variance is confounded with residual variance.")
    if n_age < 2 or n_sex < 2:
        raise ValueError("The requested age and sex variance components each require multiple observed levels.")

    # Check whether age, sex, and residual covariance components are distinguishable.
    age_kernel = np.equal.outer(data["age"].to_numpy(), data["age"].to_numpy())
    sex_kernel = np.equal.outer(data["sex"].cat.codes.to_numpy(), data["sex"].cat.codes.to_numpy())
    vc_basis = np.column_stack([np.eye(n).ravel(), age_kernel.ravel(), sex_kernel.ravel()])

    if np.linalg.matrix_rank(vc_basis) < 3:
        raise ValueError("Age, sex, and residual covariance components are not separately identifiable.")

    n_repeated = int(data.groupby("person_id", observed=True).size().gt(1).sum())
    notes = f"Exact age categories: {n_age}; sex categories: {n_sex}. No patient random intercept."
    if n_sex <= 2:
        notes += " Sex variance is estimated from only two categories."
    if n_repeated:
        notes += f" {n_repeated} patients contribute repeated courses."
    warnings.warn(notes, UserWarning, stacklevel=2)

    # Protect previous results and save the exact model inputs.
    output = Path(output_csv).expanduser()
    if output.suffix.lower() != ".csv":
        raise ValueError("output_csv must end in .csv.")

    data_path = output.with_name(f"{output.stem}_data.csv")
    metadata_path = output.with_name(f"{output.stem}_roi_metadata.csv")

    for path in (output, data_path, metadata_path):
        if path.exists() and not overwrite:
            raise FileExistsError(f"Output already exists: {path}")

    output.parent.mkdir(parents=True, exist_ok=True)
    saved_columns = list(dict.fromkeys([*common, *[c for c in ("full_file_path",) if c in df], *roi_cols]))
    df[saved_columns].to_csv(data_path, index=False)
    spec.to_csv(metadata_path, index=False)

    formula = "cgi_change ~ 1 + E_x + E_x:ef_sf"
    terms = {
        "Intercept": "intercept",
        "E_x": "E_x",
        "E_x:ef_sf": "E_x_by_ef_sf",
    }

    outputs = []
    first_write = True

    with tqdm(
        spec.to_dict("records"),
        desc="ROI mixed models",
        unit="model",
        mininterval=0,
        miniters=1,
    ) as pbar:
        for model_number, metadata in enumerate(pbar, start=1):
            pbar.set_postfix(
                roi=metadata["roi"],
                hemi=metadata["hemisphere"],
                metric=metadata["metric"],
                refresh=True,
            )

            row = {
                **metadata,
                "model_id": f"roi_{model_number:04d}",
                "formula": formula,
                "random_effects": "(1|age) + (1|sex)",
                "reml": reml,
                "optimizer": optimizer,
                "maxiter": maxiter,
                "statsmodels_version": statsmodels.__version__,
                "n_courses": n,
                "n_patients": data["person_id"].nunique(),
                "n_repeated_patients": n_repeated,
                "n_age_levels": n_age,
                "n_sex_levels": n_sex,
                "model_notes": notes,
                "status": "error",
                "converged": False,
                "llf": np.nan,
                "aic": np.nan,
                "bic": np.nan,
                "age_variance": np.nan,
                "sex_variance": np.nan,
                "residual_variance": np.nan,
                "age_variance_se": np.nan,
                "sex_variance_se": np.nan,
                "design_rank": np.nan,
                "design_condition_number": np.nan,
                "fit_seconds": np.nan,
                "warnings": "",
                "error": "",
            }

            for prefix in terms.values():
                for statistic in ("coef", "se", "z", "p", "ci_low", "ci_high"):
                    row[f"{prefix}_{statistic}"] = np.nan

            # A safe temporary predictor name avoids special characters in ROI names.
            model_data = data[[*common, metadata["roi_column"]]].rename(
                columns={metadata["roi_column"]: "E_x"}
            )

            start = time.perf_counter()
            caught_error = None

            with warnings.catch_warnings(record=True) as captured:
                warnings.simplefilter("always")

                try:
                    model = smf.mixedlm(
                        formula,
                        data=model_data,
                        groups=np.ones(n),
                        re_formula="0",
                        vc_formula={
                            "age": "0 + C(age)",
                            "sex": "0 + C(sex)",
                        },
                        missing="raise",
                    )

                    X = np.asarray(model.exog, dtype=float)
                    if not np.isfinite(X).all():
                        raise ValueError("Non-finite fixed-effects design, including E_x:ef_sf.")

                    row["design_rank"] = int(np.linalg.matrix_rank(X))
                    row["design_condition_number"] = float(np.linalg.cond(X))

                    if row["design_rank"] < X.shape[1]:
                        raise ValueError("Rank-deficient fixed effects: intercept, E_x, and/or E_x:ef_sf are aliased.")

                    fitted = model.fit(reml=reml, method=optimizer, maxiter=maxiter, disp=False)

                    row["converged"] = bool(fitted.converged)
                    row["llf"] = float(fitted.llf)
                    row["residual_variance"] = float(fitted.scale)

                    if not reml:
                        row.update(aic=float(fitted.aic), bic=float(fitted.bic))

                    # Variance-component SE convention used by MixedLM.summary().
                    vc_se = np.sqrt(fitted.scale * np.diag(np.asarray(fitted.cov_params()))[model.k_fe:])
                    for name, variance, se in zip(model.exog_vc.names, fitted.vcomp, vc_se, strict=True):
                        row[f"{name}_variance"] = float(variance)
                        row[f"{name}_variance_se"] = float(se)

                    # Fixed effects only; variance parameters have a different parameterization.
                    ci = fitted.conf_int(alpha=0.05)

                    for term, prefix in terms.items():
                        row.update({
                            f"{prefix}_coef": float(fitted.fe_params[term]),
                            f"{prefix}_se": float(fitted.bse_fe[term]),
                            f"{prefix}_z": float(fitted.tvalues[term]),
                            f"{prefix}_p": float(fitted.pvalues[term]),
                            f"{prefix}_ci_low": float(ci.loc[term, 0]),
                            f"{prefix}_ci_high": float(ci.loc[term, 1]),
                        })

                    fixed_stats = [
                        row[f"{prefix}_{statistic}"]
                        for prefix in terms.values()
                        for statistic in ("coef", "se", "z", "p")
                    ]
                    variances = [row["age_variance"], row["sex_variance"], row["residual_variance"]]

                    valid_stats = (
                        np.isfinite(fixed_stats).all()
                        and np.isfinite(variances).all()
                        and min(variances) >= 0
                        and row["residual_variance"] > 0
                    )

                    row["status"] = "ok" if valid_stats else "invalid_inference"
                    if not fitted.converged:
                        row["status"] = "not_converged"

                except (ValueError, RuntimeError, np.linalg.LinAlgError, FloatingPointError) as exc:
                    caught_error = exc
                    row["error"] = f"{type(exc).__name__}: {exc}"

            row["warnings"] = " | ".join(
                dict.fromkeys(f"{w.category.__name__}: {w.message}" for w in captured)
            )

            if row["status"] == "ok" and row["warnings"]:
                row["status"] = "warning"

            row["fit_seconds"] = time.perf_counter() - start

            # Save immediately, including failed/nonconverged models.
            pd.DataFrame([row]).to_csv(
                output,
                mode="w" if first_write else "a",
                header=first_write,
                index=False,
            )
            first_write = False
            outputs.append(row)

            if row["status"] in ("error", "invalid_inference", "not_converged") and on_error == "raise":
                raise RuntimeError(
                    f"{metadata['roi_column']}: {row['status']}; details saved in {output}"
                ) from caught_error

    return pd.DataFrame(outputs)

"""
example_usage(
    df_lmm, roi_info = _prepare_df_lmm_roi(
        fn_query,
        fn_cgi,
        threshold=95,
        step="step_4",
        model_type="skin_double",
        asf="asf_5",
        atlas_name="HCP_MMP1",
        percentile=95,
        percentile_weighting="node",
        combine_hemis=False,
        representative_session=2,
        exclude_rois=("lh.???", "rh.???"),
        subject_scan_map="subject_scan_map.json",)

    print(f"Treatment courses: {len(df_lmm)}")
    print(f"Unique patients: {df_lmm['person_id'].nunique()}")

    output_dir = Path("results/lmm/HCP_MMP1/skin_double_step_4_thr95_p95_node_separate")

    df_results = run_roi_lmm(
        df_lmm,
        roi_info,
        output_csv=output_dir / "roi_lmm_results.csv",
        metrics=("mean", "percentile"),
        reml=True,
        optimizer="lbfgs",
        maxiter=1000,
        on_error="record",
        overwrite=False,
    )

    display(df_results[
        [
            "roi",
            "hemisphere",
            "metric",
            "E_x_coef",
            "E_x_p",
            "E_x_by_ef_sf_coef",
            "E_x_by_ef_sf_p",
            "converged",
            "status",
        ]
    ])
)
"""

runner

In [ ]:
from itertools import product
from numbers import Real
from pathlib import Path
import time

import numpy as np
import pandas as pd
from tqdm import tqdm


# -------------------- Settings --------------------

atlas_name = "HCP_MMP1"
asf = "asf_5"

# None selects all available values after filtering the simulation inventory.
model_types = None  # Example: ["skin_double"]
steps = None        # Example: ["step_0", "step_4"]
thresholds = None   # Example: [95]

percentiles = [95]  # Example: [90, 95, 99]
percentile_weightings = ["area"] # also "node"
combine_hemis_options = [True] # also False
metrics_to_run = ("mean", "percentile")

representative_session = 2
exclude_rois = ("lh.???", "rh.???")
subject_scan_map = "subject_scan_map.json"

reml = True
optimizer = "lbfgs"
maxiter = 1000
on_error = "record"  # Record supported model-fitting failures and continue.

output_dir = Path("results/lmm") / atlas_name / "sweep_01"

# -------------------- Validate and enumerate --------------------

options = {
    "percentiles": percentiles,
    "percentile_weightings": percentile_weightings,
    "combine_hemis_options": combine_hemis_options,
    "metrics_to_run": metrics_to_run,
}
for name, values in options.items():
    if isinstance(values, str) or not values or len(set(values)) != len(values):
        raise ValueError(f"{name} must be a nonempty sequence of unique values.")

if any(isinstance(p, (bool, np.bool_)) or not isinstance(p, Real)
       or not np.isfinite(p) or not 0 <= p <= 100 for p in percentiles):
    raise ValueError("percentiles must contain finite numeric values in [0, 100].")
if not set(percentile_weightings).issubset({"node", "area"}):
    raise ValueError("Surface percentile_weightings must be 'node' and/or 'area'.")
if not all(isinstance(value, (bool, np.bool_)) for value in combine_hemis_options):
    raise TypeError("combine_hemis_options must contain booleans.")
if not set(metrics_to_run).issubset({"mean", "percentile"}):
    raise ValueError("metrics_to_run must contain 'mean' and/or 'percentile'.")

query = load_query_results(fn_query)
config_columns = ["model_type_dir", "step", "threshold"]
required = {*config_columns, "asf", "step_result", "subject_dir", "full_file_path"}

if not query.columns.is_unique or not required.issubset(query.columns):
    raise ValueError(f"Invalid query columns; missing: {sorted(required - set(query.columns))}")

# Match the simulation filters used by _prepare_df_lmm_roi().
available = query.loc[
    query["asf"].eq(asf)
    & query["step_result"].eq("fsavg_overlays")
    & query["subject_dir"].ne("subj-cat-010-001")
    & query["full_file_path"].str.endswith("/tdcs_uq_gpc_fsavg.msh", na=False)
].copy()

if available.empty or available[config_columns].isna().any().any():
    raise ValueError("No eligible surface simulations, or missing simulation configuration values.")
if available["threshold"].map(lambda value: isinstance(value, (bool, np.bool_))).any():
    raise ValueError("threshold must contain integers, not booleans.")

numeric_thresholds = pd.to_numeric(available["threshold"], errors="raise")
if not np.isfinite(numeric_thresholds).all() or numeric_thresholds.mod(1).ne(0).any():
    raise ValueError("threshold must contain finite integer values.")
available["threshold"] = numeric_thresholds.astype("int64")

# Apply optional restrictions without silently accepting unavailable settings.
for column, selected in (("model_type_dir", model_types), ("step", steps), ("threshold", thresholds)):
    if selected is not None:
        if isinstance(selected, str) or not selected or len(set(selected)) != len(selected):
            raise ValueError(f"Selection for {column} must be a nonempty sequence of unique values.")
        missing = set(selected) - set(available[column])
        if missing:
            raise ValueError(f"Requested {column} values unavailable under the current filters: {missing}")
        available = available.loc[available[column].isin(selected)].copy()

# only run 2 steps, 5 is too much for prelim
available = available[available['step'].isin(['step_0', 'step_2'])]

# Use combinations actually present in the inventory.
base_configs = list(
    available[config_columns].drop_duplicates().sort_values(config_columns).itertuples(index=False, name=None)
)
if not base_configs:
    raise ValueError("No simulation configurations remain.")

jobs = []

for base, percentile, weighting, combine_hemis in product(
    base_configs, percentiles, percentile_weightings, combine_hemis_options
):
    first_summary = percentile == percentiles[0] and weighting == percentile_weightings[0]
    batch_metrics = tuple(m for m in metrics_to_run if m != "mean" or first_summary)

    if not batch_metrics:
        continue  # Mean-only analysis needs just the first extraction setting.

    model_type, step, threshold = base
    jobs.append({
        "batch_id": f"batch_{len(jobs) + 1:04d}",
        "model_type": model_type,
        "step": step,
        "threshold": int(threshold),
        "percentile": float(percentile),
        "percentile_weighting": weighting,
        "combine_hemis": bool(combine_hemis),
        "metrics": batch_metrics,
    })

# Require a fresh directory so partial and completed runs cannot be mixed.
if output_dir.exists() and (not output_dir.is_dir() or any(output_dir.iterdir())):
    raise FileExistsError(f"Choose a new/empty output_dir; existing outputs found in {output_dir}")

output_dir.mkdir(parents=True, exist_ok=True)

plan = pd.DataFrame(jobs).assign(atlas=atlas_name, asf=asf)
plan["metrics"] = plan["metrics"].map("|".join)
plan.to_csv(output_dir / "sweep_plan.csv", index=False)

print(f"Simulation configurations: {len(base_configs)}")
print(f"Preparation/model batches: {len(jobs)}")


# -------------------- Prepare and fit --------------------

results, batch_records = [], []

with tqdm(total=len(jobs), desc="ROI LMM sweep", unit="batch", mininterval=0, miniters=1) as pbar:
    for job in jobs:
        progress = {
            "model": job["model_type"],
            "step": job["step"],
            "thr": job["threshold"],
            "P": job["percentile"],
            "weight": job["percentile_weighting"],
            "hemis": "bilateral" if job["combine_hemis"] else "separate",
        }

        pbar.set_postfix(**progress, status="preparing", refresh=True)
        start = time.perf_counter()

        df_lmm, roi_info = _prepare_df_lmm_roi(
            fn_query,
            fn_cgi,
            threshold=job["threshold"],
            step=job["step"],
            model_type=job["model_type"],
            asf=asf,
            atlas_name=atlas_name,
            percentile=job["percentile"],
            percentile_weighting=job["percentile_weighting"],
            combine_hemis=job["combine_hemis"],
            representative_session=representative_session,
            exclude_rois=exclude_rois,
            subject_scan_map=subject_scan_map,
        )

        prepare_seconds = time.perf_counter() - start

        # Metadata retains the correct settings for each ROI/metric.
        # Mean rows have blank percentile and percentile_weighting.
        info = roi_info.loc[roi_info["metric"].isin(job["metrics"])].copy()
        info["batch_id"] = job["batch_id"]
        info["sweep_model_id"] = [
            f"{job['batch_id']}_roi_{i:04d}" for i in range(1, len(info) + 1)
        ]

        batch_dir = output_dir / job["batch_id"]
        pbar.set_postfix(**progress, status="fitting", refresh=True)
        fit_start = time.perf_counter()

        batch_results = run_roi_lmm(
            df_lmm,
            info,
            output_csv=batch_dir / "roi_lmm_results.csv",
            metrics=job["metrics"],
            reml=reml,
            optimizer=optimizer,
            maxiter=maxiter,
            on_error=on_error,
            overwrite=False,
        )

        if len(batch_results) != len(info):
            raise ValueError(f"{job['batch_id']}: unexpected number of returned ROI models.")

        results.append(batch_results)

        batch_records.append({
            **job,
            "metrics": "|".join(job["metrics"]),
            "atlas": atlas_name,
            "asf": asf,
            "n_models": len(batch_results),
            "n_courses": len(df_lmm),
            "n_patients": df_lmm["person_id"].nunique(),
            "n_ok": int(batch_results["status"].eq("ok").sum()),
            "n_warnings": int(batch_results["status"].eq("warning").sum()),
            "n_failed": int(
                batch_results["status"].isin(["error", "not_converged", "invalid_inference"]).sum()
            ),
            "prepare_seconds": prepare_seconds,
            "fit_seconds": time.perf_counter() - fit_start,
            "results_csv": str((batch_dir / "roi_lmm_results.csv").resolve()),
        })

        pd.DataFrame(batch_records).to_csv(output_dir / "completed_batches.csv", index=False)

        pbar.update(1)
        pbar.refresh()


# -------------------- Combined results --------------------

# Written only after every planned batch has returned.
df_results = pd.concat(results, ignore_index=True)

if df_results["sweep_model_id"].duplicated().any():
    raise ValueError("Duplicate sweep model IDs.")

df_results.to_csv(output_dir / "roi_lmm_all_results.csv", index=False)

print(f"Finished {len(jobs)} batches; recorded {len(df_results)} ROI/metric models.")
print(f"Combined results: {output_dir / 'roi_lmm_all_results.csv'}")
print(df_results["status"].value_counts().to_string())

### 1.3) `Tx ~ 1 + E_global_mean + E_global_mean:ef_sf + (1|age) + (1|sex)`

In [ ]:
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e
import faulthandler
fault_log = open("kernel_fault.log", "w")
faulthandler.enable(file=fault_log, all_threads=True)

def _prepare_df_lmm_2a():
    df_query = load_query_results(fn_query)
    df_query = df_query[df_query['threshold']==95]
    df_query = df_query[df_query['subject_dir'] != 'subj-cat-010-001']
    query_filters = {"model_type_dir": "skin_double", 
                    "step_result": "fsavg_overlays",
                    "step": "step_4", 
                    "asf": "asf_5",
                    "threshold":95}

    df_cgi  = load_course_data(fn_cgi)

    df_stim = load_stimulation_data(fn_cgi)
    df_stim = _compute_ef_sf(df_stim)
    df_stim = df_stim[df_stim['subjid'] != 'subj-cat-010-001']
    
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy() # only keep subjs present in both tables

    df_stim_mapped = map_session_files(
        sessions            = df_stim, 
        query               = df_query,
        filename            = "tdcs_uq_gpc_fsavg.msh",
        query_filters       = query_filters,
        subject_scan_map    ="subject_scan_map.json",
    )
    df_stim_mapped = df_stim_mapped[df_stim_mapped['session_num']==2]

    df_mesh_roi = extract_fsaverage_roi_metrics(
        df_stim_mapped,
        path_column="full_file_path",
        field_name="magnE_mean",
        atlas_name="HCP_MMP1",
        min_nodes_per_roi=20,
        percentile=95,
        output_format="wide",
    )

    df_E_global = global_mean_e(
        (simnibs.read_msh(path) for path in df_mesh_roi["full_file_path"]),
        field_name="magnE_mean",
    )

    df_E_global = pd.DataFrame({
        "full_file_path": df_mesh_roi["full_file_path"].to_numpy(),
        "e_global": df_E_global[:, 0],
    })

    df_stim_global = df_stim_mapped.merge(
        df_E_global,
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    df_stim_global.index = df_stim_mapped.index.copy()
    shared_metadata = [
        col for col in ["age", "sex", "dob"]
            if col in df_stim_global.columns and col in df_cgi.columns
    ]
    
    df_combined = df_stim_global.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
        )
    df_combined = df_combined.rename(columns={"e_global": "E_global_mean"})
    return df_combined

def run_2a(df:pd.DataFrame):
    import numpy as np
    import statsmodels.formula.api as smf

    cols = ["cgi_change", "E_global_mean", "ef_sf", "age", "sex"]
    df = df[cols].copy()

    if df.isna().any().any():
        raise ValueError("Missing values in model columns.")

    df["sex"] = df["sex"].astype("category")

    model = smf.mixedlm(
        "cgi_change ~ 1 + E_global_mean + E_global_mean:ef_sf",
        data=df,
        groups=np.ones(len(df)),
        re_formula="0",
        vc_formula={
            "age": "0 + C(age)",
            "sex": "0 + C(sex)",
        },
    )

    result = model.fit(reml=True, method="lbfgs")
    
    print(result.summary())
    return result

df = _prepare_df_lmm_2a()
print(f"Number of treatment courses: {df.shape[0]}")
print(f"Number of unique patients: {df.person_id.unique().shape[0]}")
# result = run_2a(df)

### 1.3.b) `Tx ~ 1 + E_global_mean + E_global_mean:ef_sf + age + (1|sex)`

In [ ]:
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e
import faulthandler
fault_log = open("kernel_fault.log", "w")
faulthandler.enable(file=fault_log, all_threads=True)

def _prepare_df_lmm_13b():
    df_query = load_query_results(fn_query)
    df_query = df_query[df_query['threshold']==95]
    df_query = df_query[df_query['subject_dir'] != 'subj-cat-010-001']
    query_filters = {"model_type_dir": "skin_double", 
                    "step_result": "fsavg_overlays",
                    "step": "step_4", 
                    "asf": "asf_5",
                    "threshold":95}

    df_cgi  = load_course_data(fn_cgi)

    df_stim = load_stimulation_data(fn_cgi)
    df_stim = _compute_ef_sf(df_stim)
    df_stim = df_stim[df_stim['subjid'] != 'subj-cat-010-001']
    
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy() # only keep subjs present in both tables

    df_stim_mapped = map_session_files(
        sessions            = df_stim, 
        query               = df_query,
        filename            = "tdcs_uq_gpc_fsavg.msh",
        query_filters       = query_filters,
        subject_scan_map    ="subject_scan_map.json",
    )
    df_stim_mapped = df_stim_mapped[df_stim_mapped['session_num']==2]

    df_mesh_roi = extract_fsaverage_roi_metrics(
        df_stim_mapped,
        path_column="full_file_path",
        field_name="magnE_mean",
        atlas_name="HCP_MMP1",
        min_nodes_per_roi=20,
        percentile=95,
        output_format="wide",
    )

    df_E_global = global_mean_e(
        (simnibs.read_msh(path) for path in df_mesh_roi["full_file_path"]),
        field_name="magnE_mean",
    )

    df_E_global = pd.DataFrame({
        "full_file_path": df_mesh_roi["full_file_path"].to_numpy(),
        "e_global": df_E_global[:, 0],
    })

    df_stim_global = df_stim_mapped.merge(
        df_E_global,
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    df_stim_global.index = df_stim_mapped.index.copy()
    shared_metadata = [
        col for col in ["age", "sex", "dob"]
            if col in df_stim_global.columns and col in df_cgi.columns
    ]
    
    df_combined = df_stim_global.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
        )
    df_combined = df_combined.rename(columns={"e_global": "E_global_mean"})
    return df_combined

def run_13b(df:pd.DataFrame):
    import numpy as np
    import statsmodels.formula.api as smf

    cols = ["cgi_change", "E_global_mean", "ef_sf", "age", "sex"]
    df = df[cols].copy()

    if df.isna().any().any():
        raise ValueError("Missing values in model columns.")

    df["sex"] = df["sex"].astype("category")

    model = smf.mixedlm(
        "cgi_change ~ 1 + E_global_mean + E_global_mean:ef_sf + age",
        data=df,
        groups=np.ones(len(df)),
        re_formula="0",
        vc_formula={
            "sex": "0 + C(sex)",
        },
    )

    result = model.fit(reml=True, method="lbfgs")
    
    print(result.summary())
    return result

df = _prepare_df_lmm_13b()
print(f"Number of treatment courses: {df.shape[0]}")
print(f"Number of unique patients: {df.person_id.unique().shape[0]}")
result = run_13b(df)

### 1.3.c) `Tx ~ 1 + E_global_mean + E_global_mean:ef_sf + age + cgi_start + (1 | sex)`

In [ ]:
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e
import faulthandler
fault_log = open("kernel_fault.log", "w")
faulthandler.enable(file=fault_log, all_threads=True)

def _prepare_df_lmm_13c():
    df_query = load_query_results(fn_query)
    df_query = df_query[df_query['threshold']==95]
    df_query = df_query[df_query['subject_dir'] != 'subj-cat-010-001']
    query_filters = {"model_type_dir": "skin_double", 
                    "step_result": "fsavg_overlays",
                    "step": "step_4", 
                    "asf": "asf_5",
                    "threshold":95}

    df_cgi  = load_course_data(fn_cgi)

    df_stim = load_stimulation_data(fn_cgi)
    df_stim = _compute_ef_sf(df_stim)
    df_stim = df_stim[df_stim['subjid'] != 'subj-cat-010-001']
    
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy() # only keep subjs present in both tables

    df_stim_mapped = map_session_files(
        sessions            = df_stim, 
        query               = df_query,
        filename            = "tdcs_uq_gpc_fsavg.msh",
        query_filters       = query_filters,
        subject_scan_map    ="subject_scan_map.json",
    )
    df_stim_mapped = df_stim_mapped[df_stim_mapped['session_num']==2]

    df_mesh_roi = extract_fsaverage_roi_metrics(
        df_stim_mapped,
        path_column="full_file_path",
        field_name="magnE_mean",
        atlas_name="HCP_MMP1",
        min_nodes_per_roi=20,
        percentile=95,
        output_format="wide",
    )

    df_E_global = global_mean_e(
        (simnibs.read_msh(path) for path in df_mesh_roi["full_file_path"]),
        field_name="magnE_mean",
    )

    df_E_global = pd.DataFrame({
        "full_file_path": df_mesh_roi["full_file_path"].to_numpy(),
        "e_global": df_E_global[:, 0],
    })

    df_stim_global = df_stim_mapped.merge(
        df_E_global,
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    df_stim_global.index = df_stim_mapped.index.copy()
    shared_metadata = [
        col for col in ["age", "sex", "dob"]
            if col in df_stim_global.columns and col in df_cgi.columns
    ]
    
    df_combined = df_stim_global.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
        )
    df_combined = df_combined.rename(columns={"e_global": "E_global_mean"})
    return df_combined

def run_13c(df:pd.DataFrame):
    import numpy as np
    import statsmodels.formula.api as smf

    cols = ["cgi_change","cgi_start", "E_global_mean", "ef_sf", "age", "sex"]
    df = df[cols].copy()

    if df.isna().any().any():
        raise ValueError("Missing values in model columns.")

    df["sex"] = df["sex"].astype("category")

    model = smf.mixedlm(
        "cgi_change ~ 1 + E_global_mean + E_global_mean:ef_sf + age + cgi_start",
        data=df,
        groups=np.ones(len(df)),
        re_formula="0",
        vc_formula={
            "sex": "0 + C(sex)",
        },
    )

    result = model.fit(reml=True, method="lbfgs")
    
    print(result.summary())
    return result

df = _prepare_df_lmm_13c()
print(f"Number of treatment courses: {df.shape[0]}")
print(f"Number of unique patients: {df.person_id.unique().shape[0]}")
result = run_13c(df)

### 1.4) `Tx ~ 1 + E_95_parcel + E_95_parcel:ef_sf + (1|age) + (1|sex)`

In [ ]:
"""
1. load fsaverage meshes
2. get all ROIs
"""

from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e

df_query = load_query_results(fn_query)
df_query = df_query[df_query['threshold']==95]
df_query = df_query[df_query['subject_dir'] != 'subj-cat-010-001']
query_filters = {"model_type_dir": "skin_double", 
                 "step_result": "fsavg_overlays",
                 "step": "step_4", 
                 "asf": "asf_5",
                 "threshold":95}

df_cgi  = load_course_data(fn_cgi)

df_stim = load_stimulation_data(fn_cgi)
df_stim = df_stim[df_stim['subjid'] != 'subj-cat-010-001']

df_stim_mapped = map_session_files(
    sessions            = df_stim, 
    query               = df_query,
    filename            = "tdcs_uq_gpc_fsavg.msh",
    query_filters       = query_filters,
    subject_scan_map    ="subject_scan_map.json",
)

df_mesh_roi = extract_fsaverage_roi_metrics(
    df_stim_mapped,
    path_column="full_file_path",
    field_name="magnE_mean",
    atlas_name="HCP_MMP1",
    min_nodes_per_roi=20,
    percentile=95,
    output_format="wide",
)

# # keep percentile | mean columns only, drop other
# df_mean = df_mesh_roi.loc[
#     :, ~df_mesh_roi.columns.str.endswith("_e_percentile")
# ].copy()

# df_percentile = df_mesh_roi.loc[
#     :, ~df_mesh_roi.columns.str.endswith("_e_mean")
# ].copy()

# # Attach parcel metrics to each original session.
# df_stim_roi = join_session_roi_metrics(
#     df_stim_mapped,
#     df_mean,
#     path_column="full_file_path",
# )

# df_stim_roi = df_stim_roi[df_stim_roi['session_num']==2]
# df_stim_roi

## 2) Correlations

### Functions

In [16]:
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from itertools import product
import numpy as np
import pandas as pd
from typing import Literal
from scipy.stats import pearsonr, spearmanr

def _prepare_df_efield(threshold: int, step: str, model_type: str) -> pd.DataFrame:

    df_query = load_query_results(fn_query)
    df_query = df_query[df_query["subject_dir"] != "subj-cat-010-001"]

    query_filters = {
        "model_type_dir": model_type,
        "step_result": "fsavg_overlays",
        "step": step,
        "asf": "asf_5",
        "threshold": threshold,
    }

    df_cgi = load_course_data(fn_cgi)

    df_stim = load_stimulation_data(fn_cgi)
    df_stim = _compute_ef_sf(df_stim)
    df_stim = df_stim[df_stim["subjid"] != "subj-cat-010-001"]
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy()

    df_stim_mapped = map_session_files(
        sessions=df_stim,
        query=df_query,
        filename="tdcs_uq_gpc_fsavg.msh",
        query_filters=query_filters,
        subject_scan_map="subject_scan_map.json",
    )

    # Temporary course-level representation.
    df_stim_mapped = df_stim_mapped.loc[df_stim_mapped["session_num"] == 2].copy()

    if df_stim_mapped.empty:
        raise ValueError(f"No eligible sessions for {model_type}, {step}, threshold={threshold}")

    if df_stim_mapped["full_file_path"].isna().any():
        raise ValueError("Missing simulation file paths.")

    # Compute global E-field only once per unique mesh.
    paths = df_stim_mapped["full_file_path"].drop_duplicates().to_numpy()

    e_global = global_mean_e(
        (simnibs.read_msh(path) for path in paths),
        field_name="magnE_mean",
    )

    if e_global.shape[0] != len(paths):
        raise ValueError("E-field result count does not match the number of unique meshes.")

    df_E_global = pd.DataFrame({
        "full_file_path": paths,
        "E_global_mean": e_global[:, 0],
    })

    df_stim_global = df_stim_mapped.merge(
        df_E_global,
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    shared_metadata = [
        col for col in ["age", "sex", "dob"]
        if col in df_stim_global.columns and col in df_cgi.columns
    ]

    return df_stim_global.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
    )

def run_efield_correlation(
    df: pd.DataFrame,
    x: str = "E_global_mean",
    y: str = "cgi_change",
    method: Literal["pearson", "spearman"] = "pearson") -> dict:
    
    methods = {"pearson": pearsonr, "spearman": spearmanr}
    if method not in methods:
        raise ValueError(f"Invalid method {method!r}. Expected one of {list(methods)}")

    required = ["person_id", "series_num", x, y]
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    if df.duplicated(["person_id", "series_num"]).any():
        raise ValueError("Duplicate treatment courses detected.")

    pairs = df[[x, y]].apply(pd.to_numeric, errors="raise")
    valid = pairs.notna().all(axis=1)
    pairs = pairs.loc[valid]

    if not np.isfinite(pairs.to_numpy()).all():
        raise ValueError("Non-finite correlation values detected.")

    result = {
        "method": method,
        "x": x,
        "y": y,
        "n_courses": len(pairs),
        "n_patients": df.loc[valid, "person_id"].nunique(),
        "n_missing": len(df) - len(pairs),
        "r": np.nan,
        "p": np.nan,
        "status": "ok",
    }

    if len(pairs) < 3:
        result["status"] = "insufficient_n"
    elif (pairs.nunique() < 2).any():
        result["status"] = "constant_variable"
    else:
        corr = methods[method](pairs[x], pairs[y])
        result.update(r=float(corr.statistic), p=float(corr.pvalue))

    return result


### 2.1) Tx ~ E_global_mean

In [ ]:
from pathlib import Path

df_query = load_query_results(fn_query)

steps       = sorted(df_query["step"].dropna().unique())
model_types = sorted(df_query["model_type_dir"].dropna().unique())
thresholds  = sorted(df_query["threshold"].dropna().unique())

from itertools import product
from tqdm import tqdm
from itertools import product
import time

configs = list(product(model_types, steps, thresholds))
print(f"total combinations for config: {len(configs)}")
methods = ["pearson", "spearman"]
results = []

with tqdm( total=len(configs) * len(methods), desc="E-field correlations",
          unit="corr", mininterval=0, miniters=1) as pbar:
    
    for model_type, step, threshold in configs:
        pbar.set_postfix(model=model_type, step=step, threshold=threshold, refresh=True)
        start = time.perf_counter()

        df = _prepare_df_efield(
            threshold=int(threshold),
            step=step,
            model_type=model_type,
        )

        for method in methods:
            corr = run_efield_correlation(df, method=method)

            results.append({
                "model_type": model_type,
                "step": step,
                "threshold": threshold,
                "percentile": percentile,
                **corr,
            })

            pbar.update(1)
            pbar.refresh()

        pbar.set_postfix(last_sec=round(time.perf_counter() - start, 1), refresh=True)

df_results = pd.DataFrame(results)

fn_out = Path("results/correlations/global_efield.csv")
fn_out.parent.mkdir(parents=True, exist_ok=True)

df_results.to_csv(fn_out, index=False)

### 2.2) Tx ~ E_parcel_95 

In [13]:
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files
from simnibs_parcel_analysis.fsaverage_roi_metrics import global_mean_e
from simnibs_parcel_analysis.fsaverage_roi_metrics import (
    extract_fsaverage_roi_metrics, 
    join_session_roi_metrics
)

import simnibs
from itertools import product
import numpy as np
import pandas as pd
from typing import Literal
from scipy.stats import pearsonr, spearmanr
from pathlib import Path
from itertools import product
from tqdm.auto import tqdm

def _prepare_df_efield_roi(
    threshold: int,
    step: str,
    model_type: str,
    percentile: float = 95,
    combine_hemis: bool = False) -> pd.DataFrame:

    df_query = load_query_results(fn_query)
    df_query = df_query.loc[df_query["subject_dir"] != "subj-cat-010-001"].copy()

    query_filters = {
        "model_type_dir": model_type,
        "step_result": "fsavg_overlays",
        "step": step,
        "asf": "asf_5",
        "threshold": threshold,
    }

    df_cgi = load_course_data(fn_cgi)

    df_stim = load_stimulation_data(fn_cgi)
    df_stim = _compute_ef_sf(df_stim)
    df_stim = df_stim.loc[df_stim["subjid"] != "subj-cat-010-001"].copy()
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy()

    df_stim_mapped = map_session_files(
        sessions=df_stim,
        query=df_query,
        filename="tdcs_uq_gpc_fsavg.msh",
        query_filters=query_filters,
        subject_scan_map="subject_scan_map.json",
    )

    # Temporary course-level representation.
    df_stim_mapped = df_stim_mapped.loc[df_stim_mapped["session_num"] == 2].copy()

    if df_stim_mapped.empty:
        raise ValueError(f"No eligible sessions for {model_type}, {step}, threshold={threshold}")

    if df_stim_mapped["full_file_path"].isna().any():
        raise ValueError("Missing simulation file paths.")

    # Extract ROI metrics once per unique mesh.
    df_unique = df_stim_mapped.drop_duplicates(subset=["full_file_path"]).copy()

    df_mesh_roi = extract_fsaverage_roi_metrics(
        df_unique,
        path_column="full_file_path",
        field_name="magnE_mean",
        atlas_name="HCP_MMP1",
        min_nodes_per_roi=20,
        percentile=percentile,
        output_format="wide",
    )

    # Identify hemisphere-specific ROI columns.
    roi_cols = [
        col for col in df_mesh_roi.columns
        if isinstance(col, str) and col.startswith(("lh.", "rh."))
    ]

    if not roi_cols:
        raise ValueError("No lh./rh. ROI columns found in extracted metrics.")

    df_roi = df_mesh_roi[["full_file_path", *roi_cols]].copy()

    if df_roi["full_file_path"].isna().any() or df_roi["full_file_path"].duplicated().any():
        raise ValueError("ROI metrics must contain one row per unique, non-missing mesh path.")

    # Optionally combine corresponding left/right ROIs.
    if combine_hemis:
        lh = {col[3:]: col for col in roi_cols if col.startswith("lh.")}
        rh = {col[3:]: col for col in roi_cols if col.startswith("rh.")}

        if set(lh) != set(rh):
            raise ValueError(f"Unmatched hemisphere ROIs: {sorted(set(lh) ^ set(rh))}")

        bilateral = pd.DataFrame({
            roi: (df_roi[lh[roi]] + df_roi[rh[roi]]) / 2
            for roi in sorted(lh)
        }, index=df_roi.index)

        df_roi = pd.concat([df_roi[["full_file_path"]], bilateral], axis=1)

    # Merge ROI metrics back to treatment sessions.
    overlap = (set(df_roi.columns) & set(df_stim_mapped.columns)) - {"full_file_path"}
    if overlap:
        raise ValueError(f"ROI columns conflict with session columns: {sorted(overlap)}")

    df_stim_roi = df_stim_mapped.merge(
        df_roi,
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
    )

    shared_metadata = [
        col for col in ["age", "sex", "dob"]
        if col in df_stim_roi.columns and col in df_cgi.columns
    ]

    return df_stim_roi.drop(columns=shared_metadata).merge(
        df_cgi,
        on=["person_id", "series_num"],
        how="left",
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
    )

from typing import Literal
import pandas as pd


def run_roi_correlations(
    df: pd.DataFrame,
    combine_hemis: bool = False,
    metric: Literal["mean", "percentile"] = "percentile",
    method: Literal["pearson", "spearman"] = "pearson",
) -> pd.DataFrame:

    if metric not in ("mean", "percentile"):
        raise ValueError(f"Invalid metric: {metric!r}")

    suffix = f"_e_{metric}"

    roi_cols = [
        col for col in df.columns
        if isinstance(col, str)
        and col.endswith(suffix)
        and (not col.startswith(("lh.", "rh.")) if combine_hemis else col.startswith(("lh.", "rh.")))
        and "???" not in col
    ]

    if not roi_cols:
        raise ValueError(f"No ROI columns found for combine_hemis={combine_hemis}, metric={metric}")

    results = []

    for col in roi_cols:
        corr = run_efield_correlation(df, x=col, y="cgi_change", method=method)

        hemisphere = col.split(".", 1)[0] if not combine_hemis else "bilateral"
        roi = col.split(".", 1)[-1].removesuffix(suffix)

        results.append({
            "roi": roi,
            "hemisphere": hemisphere,
            "roi_column": col,
            "metric": metric,
            "combine_hemis": combine_hemis,
            **corr,
        })

    return pd.DataFrame(results)

In [ ]:
from itertools import product
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import time

df_query = load_query_results(fn_query)

# steps       = sorted(df_query["step"].dropna().unique())
model_types = sorted(df_query["model_type_dir"].dropna().unique())
thresholds  = sorted(df_query["threshold"].dropna().unique())
steps       = ['step_0','step_2', 'step_4']

percentiles = [95]  # supports multiple values, e.g. [90, 95, 99].
methods = ["pearson", "spearman"]
combine_hemis_options = [True, False]

if not percentiles or len(percentiles) != len(set(percentiles)):
    raise ValueError("percentiles must be nonempty and contain no duplicate values.")

configs = list(product(model_types, steps, thresholds, percentiles, combine_hemis_options))
results = []

# Per simulation/hemisphere configuration:
# one mean analysis + one analysis for each requested percentile, for each method.
n_base_configs = len(model_types) * len(steps) * len(thresholds) * len(combine_hemis_options)
total_batches = n_base_configs * (1 + len(percentiles)) * len(methods)

if not configs:
    raise ValueError("No simulation configurations to process.")

print(f"Total preparation configurations: {len(configs)}")
print(f"Total correlation batches: {total_batches}")

with tqdm(total=total_batches, desc="ROI correlations", unit="batch", mininterval=0, miniters=1) as pbar:
    for model_type, step, threshold, percentile, combine_hemis in configs:
        progress = {
            "model": model_type,
            "step": step,
            "threshold": threshold,
            "percentile": percentile,
            "hemis": combine_hemis,
        }

        pbar.set_postfix(**progress, status="preparing", refresh=True)
        start = time.perf_counter()

        # Both ROI means and the requested ROI percentiles are extracted here.
        df = _prepare_df_efield_roi(
            threshold=int(threshold),
            step=step,
            model_type=model_type,
            percentile=percentile,
            combine_hemis=combine_hemis,
        )

        # Analyze means only during the first percentile pass to avoid duplicate tests.
        metrics = ["mean", "percentile"] if percentile == percentiles[0] else ["percentile"]

        for metric, method in product(metrics, methods):
            pbar.set_postfix(**progress, metric=metric, method=method, status="correlating", refresh=True)

            corr = run_roi_correlations(
                df,
                combine_hemis=combine_hemis,
                metric=metric,
                method=method,
            )

            corr["model_type"] = model_type
            corr["step"] = step
            corr["threshold"] = threshold
            corr["metric"] = metric

            # Percentile is an analysis parameter only for the percentile metric.
            corr["percentile"] = float(percentile) if metric == "percentile" else float("nan")

            results.append(corr)

            pbar.update(1)
            pbar.refresh()

        elapsed = round(time.perf_counter() - start, 1)
        pbar.set_postfix(**progress, status="done", last_sec=elapsed, refresh=True)

df_results = pd.concat(results, ignore_index=True)
df_results['r_abs'] = df_results[r]
output = Path("results/correlations/roi_efield.csv")
output.parent.mkdir(parents=True, exist_ok=True)
df_results.to_csv(output, index=False)

Save the results to .msh file

In [ ]:
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
import re

def write_all_roi_correlation_meshes(
    results: pd.DataFrame,
    template_path: str | Path,
    output_dir: str | Path,
    *,
    atlas_name: str = "HCP_MMP1",
    value_columns: tuple[str, ...] = ("r", "r_abs", "p", "n_courses"),
    overwrite: bool = False,
) -> pd.DataFrame:
    """Export one mesh/source CSV per analysis present in results; return an export index."""
    if results.empty or not results.columns.is_unique:
        raise ValueError("Results must be nonempty and have unique column names.")
    if isinstance(value_columns, str) or not value_columns or len(set(value_columns)) != len(value_columns):
        raise ValueError("value_columns must be a nonempty sequence of unique column names.")

    config_cols = ["model_type", "step", "threshold", "percentile", "combine_hemis", "metric", "method", "y"]
    optional_config = [c for c in ("asf", "atlas", "field_name") if c in results.columns]
    group_cols = config_cols + optional_config

    required = {*group_cols, "roi", "hemisphere", "status"}
    missing = required - set(results.columns)
    if missing:
        raise ValueError(f"Missing result columns: {sorted(missing)}")

    frame = results.copy()

    # Derive absolute correlation only if requested and not already present.
    if "r_abs" in value_columns and "r_abs" not in frame.columns:
        if "r" not in frame.columns:
            raise ValueError("Cannot calculate r_abs without an r column.")
        frame["r_abs"] = pd.to_numeric(frame["r"], errors="raise").abs()

    missing = set(value_columns) - set(frame.columns)
    if missing:
        raise ValueError(f"Missing value columns: {sorted(missing)}")

    if not frame["combine_hemis"].map(lambda x: isinstance(x, (bool, np.bool_))).all():
        raise TypeError("combine_hemis must contain booleans, not strings or integers.")
    if frame[[c for c in group_cols if c != "percentile"]].isna().any().any():
        raise ValueError("Configuration values cannot be missing, except percentile for non-percentile metrics.")
    if "atlas" in frame.columns and not frame["atlas"].eq(atlas_name).all():
        raise ValueError("All results must match atlas_name; filter by atlas before exporting.")
    if frame.duplicated(group_cols + ["hemisphere", "roi"]).any():
        raise ValueError("Duplicate ROI results within an analysis; do not silently deduplicate them.")

    # Retain mean analyses, whose percentile values are NaN.
    grouped = frame.groupby(group_cols, dropna=False, observed=True, sort=True)

    output_dir = Path(output_dir).expanduser()
    index_path = output_dir / "export_index.csv"

    if index_path.exists() and not overwrite:
        raise FileExistsError(f"Output already exists: {index_path}")

    records = []
    used_paths = set()

    with tqdm(total=grouped.ngroups, desc="Exporting ROI maps", unit="mesh", mininterval=0, miniters=1) as pbar:
        for key, selected in grouped:
            config = dict(zip(group_cols, key, strict=True))
            hemi_label = "bilateral" if config["combine_hemis"] else "separate"

            metric_label = config["metric"]
            if metric_label == "percentile":
                percentile = float(config["percentile"])
                if not np.isfinite(percentile) or not 0 <= percentile <= 100:
                    raise ValueError(f"Invalid percentile in configuration: {config}")
                metric_label = f"p{percentile:g}"
            elif pd.notna(config["percentile"]):
                raise ValueError("Non-percentile metrics must have percentile=NaN.")

            # Include the outcome and optional settings to distinguish analyses.
            parts = [
                atlas_name, str(config["y"]), str(config["model_type"]), str(config["step"]),
                f"thr{float(config['threshold']):g}", metric_label, str(config["method"]), hemi_label,
            ]
            parts.extend(f"{c}-{config[c]}" for c in optional_config if c != "atlas")
            filename = re.sub(r"[^A-Za-z0-9_.-]+", "-", "_".join(parts)) + ".msh"
            output_path = output_dir / filename

            # Even with overwrite=True, different analyses must not share a filename.
            if output_path in used_paths:
                raise ValueError(f"Different analyses produced the same filename: {output_path}")
            used_paths.add(output_path)

            pbar.set_postfix(
                model=config["model_type"],
                step=config["step"],
                threshold=config["threshold"],
                metric=metric_label,
                method=config["method"],
                hemis=hemi_label,
                refresh=True,
            )

            output = write_roi_correlation_mesh(
                selected,
                template_path=template_path,
                output_path=output_path,
                atlas_name=atlas_name,
                value_columns=value_columns,
                overwrite=overwrite,
            )

            records.append({
                **config,
                "mesh_atlas": atlas_name,
                "n_roi_results": len(selected),
                "n_ok": int(selected["status"].eq("ok").sum()),
                "mesh_path": str(output.resolve()),
                "source_csv": str(output.with_suffix(".csv").resolve()),
            })

            pbar.update(1)
            pbar.refresh()

    export_index = pd.DataFrame(records)
    export_index.to_csv(index_path, index=False)

    return export_index


template_path = Path("/projects/p32903/Alex2/datasets/freesurfer_templates/tdcs_uq_gpc_fsavg.msh")
output_dir = Path("results/correlations/maps")

df_exports = write_all_roi_correlation_meshes(
    df_results,
    template_path=template_path,
    output_dir=output_dir,
    atlas_name="HCP_MMP1",
    value_columns=("r", "r_abs", "p", "n_courses"),
    overwrite=False,
)

print(f"Exported {len(df_exports)} meshes.")
print(f"Export index: {output_dir / 'export_index.csv'}")

display(df_exports[
    ["model_type", "step", "threshold", "metric", "percentile", "method",
     "combine_hemis", "n_roi_results", "n_ok", "mesh_path"]
])

### 2.3)  FreeSurfer Desikan–Killiany + aseg (aparc+aseg.mgz)

In [ ]:
from numbers import Real
from typing import Literal

import numpy as np
import pandas as pd

from simnibs_parcel_analysis.atlas_volume import build_volume_roi_summary
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files

import importlib
import simnibs_parcel_analysis.atlas_volume as atlas_volume

importlib.reload(atlas_volume)
build_volume_roi_summary = atlas_volume.build_volume_roi_summary

def _volume_roi_name(name: str) -> str:
    """Normalize lateralized FreeSurfer labels to the existing lh./rh. ROI format."""
    prefixes = {
        "ctx-lh-": "lh.", "ctx-rh-": "rh.",
        "ctx_lh_": "lh.", "ctx_rh_": "rh.",
        "Left-": "lh.", "Right-": "rh.",
    }
    for prefix, replacement in prefixes.items():
        if name.startswith(prefix):
            return replacement + name[len(prefix):]
    raise ValueError(f"Unsupported or nonlateralized FreeSurfer ROI: {name!r}")

def _prepare_df_efield_volume(
    threshold: int,
    step: str,
    model_type: str,
    percentile: float = 95,
    combine_hemis: bool = False,
    percentile_weighting: Literal["unweighted", "spatial_weighted"] = "spatial_weighted",
    negative_atol: float = 1e-6,
) -> pd.DataFrame:
    """Prepare session-2 courses with aparc+aseg ROI means, percentiles, and CGI outcomes.

    Uses existing notebook globals: fn_query, fn_cgi, load_query_results,
    load_course_data, load_stimulation_data, and _compute_ef_sf.
    load_query_results must return dataset_root and fs_subjects_dir.
    Simulation paths must match df_query.full_file_path exactly.
    """
    if (isinstance(percentile, (bool, np.bool_)) or not isinstance(percentile, Real)
            or not np.isfinite(percentile) or not 0 <= percentile <= 100):
        raise ValueError("percentile must be a finite number in [0, 100].")
    if not isinstance(combine_hemis, (bool, np.bool_)):
        raise TypeError("combine_hemis must be a boolean.")
    if percentile_weighting not in {"unweighted", "spatial_weighted"}:
        raise ValueError("percentile_weighting must be 'unweighted' or 'spatial_weighted'.")

    # The updated loader supplies dataset_root -> fs_subjects_dir.
    df_query = load_query_results(fn_query)
    query_columns = ["full_file_path", "subject_dir", "dataset_root", "fs_subjects_dir"]
    missing = set(query_columns) - set(df_query.columns)
    if missing:
        raise ValueError(f"Missing query columns: {sorted(missing)}")
    df_query = df_query.loc[df_query["subject_dir"] != "subj-cat-010-001"].copy()

    query_filters = {
        "model_type_dir": model_type,
        "step": step,
        "asf": "asf_5",
        "threshold": threshold,
    }

    df_cgi = load_course_data(fn_cgi)
    df_stim = _compute_ef_sf(load_stimulation_data(fn_cgi))
    df_stim = df_stim.loc[df_stim["subjid"] != "subj-cat-010-001"].copy()
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy()

    # No fsavg_overlays filter: these are volumetric simulation files.
    mapped = map_session_files(
        sessions=df_stim,
        query=df_query,
        filename="tdcs_uq_gpc.hdf5",
        query_filters=query_filters,
        subject_scan_map="subject_scan_map.json",
    )

    missing = {"full_file_path", "session_num"} - set(mapped.columns)
    if missing:
        raise ValueError(f"Missing mapped columns: {sorted(missing)}")

    # Preserve the current representative-session definition.
    mapped = mapped.loc[mapped["session_num"] == 2].copy()
    if mapped.empty:
        raise ValueError(f"No eligible session 2 rows for {model_type}, {step}, threshold={threshold}")
    if mapped["full_file_path"].isna().any():
        raise ValueError("Missing simulation file paths in mapped sessions.")

    # Recover metadata by exact simulation path, not clinical subject ID.
    lookup = df_query[query_columns].drop_duplicates().rename(columns={"subject_dir": "modeled_subjid"})
    if lookup.isna().any().any():
        raise ValueError("Missing simulation or FreeSurfer metadata in the query lookup.")
    if lookup["fs_subjects_dir"].astype(str).str.strip().eq("").any():
        raise ValueError("Blank FreeSurfer SUBJECTS_DIR in the query lookup.")
    if lookup["full_file_path"].duplicated().any():
        raise ValueError("A query file path has conflicting subject, dataset, or FreeSurfer metadata.")

    metadata_cols = ["modeled_subjid", "dataset_root", "fs_subjects_dir"]
    temporary = {col: f"_query_{col}" for col in metadata_cols}
    reserved = {*temporary.values(), "_query_match"} & set(mapped.columns)
    if reserved:
        raise ValueError(f"Reserved metadata columns already exist: {sorted(reserved)}")

    mapped = mapped.merge(
        lookup.rename(columns=temporary),
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
        indicator="_query_match",
    )

    unmatched = mapped["_query_match"].ne("both")
    if unmatched.any():
        paths = mapped.loc[unmatched, "full_file_path"].unique().tolist()
        raise ValueError(f"Mapped paths do not exactly match df_query.full_file_path: {paths}")

    # Retain matching existing metadata, fill missing metadata, and reject conflicts.
    for col in metadata_cols:
        expected = mapped.pop(temporary[col])
        if col in mapped.columns:
            present = mapped[col].notna()
            if mapped.loc[present, col].ne(expected.loc[present]).any():
                raise ValueError(f"Mapped {col!r} conflicts with metadata from the query inventory.")
        mapped[col] = expected
    mapped = mapped.drop(columns="_query_match")

    # Catch duplicate/missing course identifiers before reading meshes.
    course_keys = ["person_id", "series_num"]
    for label, table in (("Mapped sessions", mapped), ("CGI table", df_cgi)):
        missing = set(course_keys) - set(table.columns)
        if missing:
            raise ValueError(f"{label} is missing course identifiers: {sorted(missing)}")
        if table[course_keys].isna().any().any():
            raise ValueError(f"{label} contains missing course identifiers.")
        if table.duplicated(course_keys).any():
            raise ValueError(f"{label} contains duplicate person_id/series_num combinations.")

    # Repeated courses can reuse one simulation; extract each unique file once.
    inventory = mapped[["full_file_path", *metadata_cols]].drop_duplicates()
    if inventory["full_file_path"].duplicated().any():
        raise ValueError("One simulation file has multiple metadata assignments.")

    roi_frames = []
    for fs_subjects_dir, group in inventory.groupby("fs_subjects_dir", sort=True, observed=True):
        paths = group["full_file_path"].tolist()
        summary = build_volume_roi_summary(
            hdf5_files=paths,
            subjects_dir=fs_subjects_dir,
            subject_ids=group["modeled_subjid"].tolist(),
            simulation_ids=paths,
            atlas_name="aparc",
            field_name="magnE_mean",
            percentile=float(percentile),
            min_elements_per_roi=20,
            freesurfer_lut='/projects/p32903/Alex2/datasets/freesurfer_templates/FreeSurferColorLUT.txt',
            negative_atol=negative_atol,

        )

        if summary.parcel_mean_e is None:
            raise ValueError(f"No ROI means returned for {fs_subjects_dir}")

        means = summary.parcel_mean_e
        percentiles = summary.p95(percentile_weighting)
        if not means.index.equals(percentiles.index) or not means.columns.equals(percentiles.columns):
            raise ValueError("ROI mean and percentile tables do not have matching simulations and ROIs.")
        if means.index.has_duplicates or set(means.index) != set(paths):
            raise ValueError("Extracted simulation IDs do not match the requested unique file paths.")

        means = means.rename(columns=lambda col: f"{_volume_roi_name(col)}_e_mean")
        percentiles = percentiles.rename(columns=lambda col: f"{_volume_roi_name(col)}_e_percentile")
        group_roi = pd.concat([means, percentiles], axis=1)
        group_roi = group_roi.rename_axis("full_file_path").reset_index()

        if not group_roi.columns.is_unique:
            raise ValueError("Duplicate ROI column names after FreeSurfer label normalization.")
        if roi_frames and not group_roi.columns.equals(roi_frames[0].columns):
            raise ValueError(f"ROI columns differ across FreeSurfer datasets: {fs_subjects_dir}")
        roi_frames.append(group_roi)

    df_roi = pd.concat(roi_frames, ignore_index=True)
    if df_roi["full_file_path"].duplicated().any():
        raise ValueError("Duplicate simulation paths in extracted ROI metrics.")

    if combine_hemis:
        lh = {col[3:]: col for col in df_roi.columns if col.startswith("lh.")}
        rh = {col[3:]: col for col in df_roi.columns if col.startswith("rh.")}
        if not lh or set(lh) != set(rh):
            raise ValueError(f"Unmatched hemisphere ROIs: {sorted(set(lh) ^ set(rh))}")

        # Average matching summaries; this is not a pooled bilateral percentile.
        bilateral = pd.DataFrame({
            roi: (df_roi[lh[roi]] + df_roi[rh[roi]]) / 2
            for roi in sorted(lh)
        }, index=df_roi.index)
        df_roi = pd.concat([df_roi[["full_file_path"]], bilateral], axis=1)

    roi_columns = df_roi.columns.drop("full_file_path").tolist()
    for label, table in (("mapped sessions", mapped), ("CGI table", df_cgi)):
        overlap = set(roi_columns) & set(table.columns)
        if overlap:
            raise ValueError(f"ROI columns conflict with {label}: {sorted(overlap)}")

    df_stim_roi = mapped.merge(df_roi, on="full_file_path", how="left", sort=False, validate="many_to_one")
    if df_stim_roi[roi_columns].isna().any().any():
        raise ValueError("Missing ROI metrics after merging simulation files.")

    shared_metadata = [col for col in ("age", "sex", "dob") if col in df_stim_roi and col in df_cgi]
    return df_stim_roi.drop(columns=shared_metadata).merge(
        df_cgi,
        on=course_keys,
        how="left",
        sort=False,
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
    )


"""Run in the existing analysis notebook after defining the preparation/correlation helpers.

Required notebook definitions:
    fn_query, fn_cgi, load_query_results, _prepare_df_efield_volume,
    run_roi_correlations (and its run_efield_correlation dependency).

The preparation helper must use freesurfer_lut=<path>, not lut=<path>.
This script does not change the extraction, cohort selection, or correlation estimator.
"""

from datetime import datetime
from hashlib import sha256
from itertools import product
from numbers import Real
from pathlib import Path
import time

import numpy as np
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

def _save_correlation_workbook(tables: dict[str, pd.DataFrame], output: Path) -> None:
    """Write tables with filters, frozen headers, and display-only numeric formatting."""
    temporary = output.with_name(f"{output.stem}.tmp.xlsx")
    with pd.ExcelWriter(temporary, engine="openpyxl") as writer:
        for name, table in tables.items():
            table.to_excel(writer, sheet_name=name, index=False, na_rep="")
            sheet = writer.sheets[name]
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            sheet.row_dimensions[1].height = 30

            for cell in sheet[1]:
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="234E70")
                cell.alignment = Alignment(wrap_text=True, vertical="center")

            for j, column in enumerate(table.columns, start=1):
                width = min(36, max(14, len(column) + 2))
                if column in {"roi", "roi_column", "x", "config_label"}:
                    width = 38
                sheet.column_dimensions[get_column_letter(j)].width = width
                number_format = "0.000E+00" if column in {"p", "p_bh_sweep"} else "0.000"
                if column in {"r", "r_abs", "p", "p_bh_sweep", "prepare_seconds"}:
                    for cells in sheet.iter_rows(min_row=2, min_col=j, max_col=j):
                        cells[0].number_format = number_format

            if name == "settings":
                sheet.column_dimensions["A"].width = 30
                sheet.column_dimensions["B"].width = 60
                for cells in sheet.iter_rows(min_row=2):
                    cells[1].alignment = Alignment(wrap_text=True, vertical="top")
                    sheet.row_dimensions[cells[0].row].height = 45

    temporary.replace(output)

# Keep these atlas/ASF labels consistent with the current preparation function.
atlas_name = "aparc"
asf = "asf_5"
percentiles = [95]  # For example: [90, 95, 99].
percentile_weightings = [ "spatial_weighted"] # also "unweighted"
methods = ["spearman"] # also "pearson"
combine_hemis_options = [False, False] # also False

output_dir = Path("results/correlations") / atlas_name
output_dir.mkdir(parents=True, exist_ok=True)
xlsx_path = output_dir / "roi_efield.xlsx"
csv_path = output_dir / "roi_efield.csv"
checkpoint_path = output_dir / "roi_efield_checkpoint.csv"

# Avoid mixing a new run with an older run or partial checkpoint.
for path in (xlsx_path, csv_path, checkpoint_path):
    if path.exists():
        raise FileExistsError(f"Output already exists; use a fresh folder or move the previous run: {path}")

for name, values in (
    ("percentiles", percentiles), ("percentile_weightings", percentile_weightings),
    ("methods", methods), ("combine_hemis_options", combine_hemis_options),
):
    if not values or len(set(values)) != len(values):
        raise ValueError(f"{name} must be nonempty and contain no duplicates.")
if any(isinstance(p, (bool, np.bool_)) or not isinstance(p, Real)
       or not np.isfinite(p) or not 0 <= p <= 100 for p in percentiles):
    raise ValueError("percentiles must contain finite numbers in [0, 100].")
if not set(percentile_weightings) <= {"unweighted", "spatial_weighted"}:
    raise ValueError("Unsupported percentile weighting.")
if not set(methods) <= {"pearson", "spearman"}:
    raise ValueError("Unsupported correlation method.")
if not all(isinstance(value, (bool, np.bool_)) for value in combine_hemis_options):
    raise TypeError("combine_hemis_options must contain booleans.")

# Select all simulation configurations actually present for the fixed filters.
df_query = load_query_results(fn_query)
config_columns = ["model_type_dir", "step", "threshold"]
required = {*config_columns, "asf", "subject_dir", "full_file_path"}
missing = required - set(df_query.columns)
if missing:
    raise ValueError(f"Missing query columns: {sorted(missing)}")

available = df_query.loc[
    df_query["asf"].eq(asf)
    & df_query["subject_dir"].ne("subj-cat-010-001")
    & df_query["full_file_path"].str.endswith("/tdcs_uq_gpc.hdf5", na=False)
].copy()

# only keep these for now, then run on HPC to include all 
keep_steps = ['step_0', 'step_2', 'step_4']
available  = available[available['step'].isin(keep_steps)]

if available.empty or available[config_columns].isna().any().any():
    raise ValueError("No eligible HDF5 configurations, or missing simulation configuration values.")
threshold_values = pd.to_numeric(available["threshold"], errors="raise")
if (available["threshold"].map(lambda value: isinstance(value, (bool, np.bool_))).any()
        or not np.isfinite(threshold_values).all() or threshold_values.mod(1).ne(0).any()):
    raise ValueError("Simulation thresholds must be finite integers.")
available["threshold"] = threshold_values.astype(int)

base_configs = list(
    available[config_columns].drop_duplicates().sort_values(config_columns).itertuples(index=False, name=None)
)
configs = [
    (*base, percentile, combine_hemis, weighting)
    for base, percentile, combine_hemis, weighting in product(
        base_configs, percentiles, combine_hemis_options, percentile_weightings
    )
]

# Means run once per simulation/hemisphere/method, not once per percentile/weighting.
total_batches = (len(base_configs) * len(combine_hemis_options) * len(methods)
                 * (1 + len(percentiles) * len(percentile_weightings)))
results, analysis_records = [], []
started_at = datetime.now().astimezone().isoformat(timespec="seconds")

print(f"Simulation configurations: {len(base_configs)}")
print(f"Preparation calls: {len(configs)}")
print(f"Correlation batches: {total_batches}")

with tqdm(total=total_batches, desc="aparc ROI correlations", unit="batch", mininterval=0, miniters=1) as pbar:
    for model_type, step, threshold, percentile, combine_hemis, weighting in configs:
        progress = {
            "model": model_type, "step": step, "threshold": threshold,
            "percentile": percentile, "weighting": weighting, "hemis": combine_hemis,
        }
        pbar.set_postfix(**progress, status="preparing", refresh=True)
        start = time.perf_counter()

        df = _prepare_df_efield_volume(
            threshold=int(threshold),
            step=step,
            model_type=model_type,
            percentile=percentile,
            combine_hemis=combine_hemis,
            percentile_weighting=weighting,
        )
        prepare_seconds = time.perf_counter() - start

        # This identifies the PREPARED course set; n_courses still reports usable pairs per ROI.
        course_keys = df[["person_id", "series_num"]].sort_values(["person_id", "series_num"])
        prepared_cohort_id = sha256(course_keys.to_csv(index=False).encode()).hexdigest()[:16]
        dataset_roots = ";".join(sorted(df["dataset_root"].unique()))
        first_summary = percentile == percentiles[0] and weighting == percentile_weightings[0]
        metrics = ["mean", "percentile"] if first_summary else ["percentile"]

        for metric, method in product(metrics, methods):
            pbar.set_postfix(**progress, metric=metric, method=method, status="correlating", refresh=True)
            corr = run_roi_correlations(df, combine_hemis=combine_hemis, metric=metric, method=method).copy()
            if corr.empty:
                raise ValueError(f"No ROI correlation results for {progress}, metric={metric}, method={method}")

            metric_label = f"p{float(percentile):g}_{weighting}" if metric == "percentile" else "mean"
            hemis_label = "bilateral" if combine_hemis else "separate"
            config_label = f"{model_type}|{step}|thr{threshold}|{metric_label}|{hemis_label}|{method}"
            metadata = {
                "analysis_id": len(analysis_records) + 1,
                "config_label": config_label,
                "atlas": atlas_name,
                "atlas_file": "aparc+aseg.mgz",
                "asf": asf,
                "model_type": model_type,
                "step": step,
                "threshold": int(threshold),
                "metric": metric,
                "percentile": float(percentile) if metric == "percentile" else np.nan,
                "percentile_weighting": weighting if metric == "percentile" else pd.NA,
                "mean_weighting": "unweighted_elements" if metric == "mean" else pd.NA,
                "combine_hemis": bool(combine_hemis),
                "hemisphere_aggregation": "mean_of_hemisphere_summaries" if combine_hemis else "none",
                "method": method,
                "field_name": "magnE_mean",
                "representative_session": 2,
                "dataset_roots": dataset_roots,
                "prepared_cohort_id": prepared_cohort_id,
            }
            corr = corr.assign(**metadata, r_abs=corr["r"].abs())

            # Persist completed batches before the expensive next preparation call.
            corr.to_csv(checkpoint_path, mode="a", header=not checkpoint_path.exists(), index=False)
            results.append(corr)
            analysis_records.append({
                **metadata,
                "n_roi_results": len(corr),
                "n_ok": int(corr["status"].eq("ok").sum()),
                "n_courses_min": int(corr["n_courses"].min()),
                "n_courses_max": int(corr["n_courses"].max()),
                "n_patients_min": int(corr["n_patients"].min()),
                "n_patients_max": int(corr["n_patients"].max()),
                "prepare_seconds": prepare_seconds,
            })
            pbar.update(1)
            pbar.refresh()

        pbar.set_postfix(**progress, status="done", last_sec=round(time.perf_counter() - start, 1), refresh=True)

if len(analysis_records) != total_batches:
    raise RuntimeError("Completed correlation batch count does not match the planned count.")

df_results = pd.concat(results, ignore_index=True)
df_analyses = pd.DataFrame(analysis_records)
test_keys = [
    "atlas", "asf", "model_type", "step", "threshold", "metric", "percentile",
    "percentile_weighting", "combine_hemis", "method", "hemisphere", "roi", "y",
]
if df_results.duplicated(test_keys).any():
    raise ValueError("Duplicate ROI tests in the final results; do not silently deduplicate them.")

# Exploratory BH adjustment across ALL valid tests in this completed sweep.
# This does not correct invalid/unclustered input p-values.
df_results["p_bh_sweep"] = np.nan
valid = df_results["status"].eq("ok")
p_values = pd.to_numeric(df_results.loc[valid, "p"], errors="raise")
if not np.isfinite(p_values).all() or not p_values.between(0, 1).all():
    raise ValueError("An 'ok' correlation has an invalid p-value.")
if valid.any():
    df_results.loc[valid, "p_bh_sweep"] = multipletests(p_values, method="fdr_bh")[1]

first_columns = [
    "analysis_id", "config_label", "atlas", "model_type", "step", "threshold", "metric",
    "percentile", "percentile_weighting", "mean_weighting", "combine_hemis", "hemisphere", "roi",
    "method", "r", "r_abs", "p", "p_bh_sweep", "n_courses", "n_patients", "n_missing", "status",
]
df_results = df_results[first_columns + [c for c in df_results.columns if c not in first_columns]]

settings = {
    "started_at": started_at,
    "completed_at": datetime.now().astimezone().isoformat(timespec="seconds"),
    "query_file": str(Path(fn_query).resolve()),
    "clinical_file": str(Path(fn_cgi).resolve()),
    "atlas": atlas_name,
    "atlas_file": "aparc+aseg.mgz",
    "simulation_selection": "All existing HDF5 model/step/threshold combinations for asf_5, excluding subj-cat-010-001",
    "percentiles": str(percentiles),
    "percentile_weightings": str(percentile_weightings),
    "methods": str(methods),
    "combine_hemis_options": str(combine_hemis_options),
    "representative_session": "2; unchanged from the preparation function",
    "min_elements_per_roi": "20; unchanged from the preparation function",
    "outcome": "cgi_change as supplied by the clinical loader; not recomputed",
    "mean_definition": "Unweighted arithmetic mean across tetrahedral elements; analyzed only once per configuration",
    "percentile_definitions": "unweighted: equal elements; spatial_weighted: tetrahedron-volume-weighted empirical CDF",
    "bilateral_definition": "Arithmetic mean of left/right summary values, not a pooled bilateral percentile",
    "mean_parameter_blanks": "percentile and percentile_weighting are blank because neither defines the ROI mean",
    "prepared_cohort_id": "Hash of sorted prepared person_id/series_num pairs; not a hash of all ROI-specific usable pairs",
    "multiplicity": "p_bh_sweep: BH across all finite status=ok tests in this completed atlas sweep",
    "p_value_limitation": "Existing two-sided Pearson/Spearman tests are unchanged; no patient-cluster correction",
    "checkpoint": "Completed batches only; no full-sweep BH adjustment and no automatic resume",
}
df_settings = pd.DataFrame(settings.items(), columns=["setting", "value"])

df_results.to_csv(csv_path, index=False)
_save_correlation_workbook({"results": df_results, "analyses": df_analyses, "settings": df_settings}, xlsx_path)

print(f"Saved {len(df_results)} ROI results from {len(df_analyses)} analysis batches.")
print(f"Excel: {xlsx_path}")
print(f"CSV: {csv_path}")
print(f"Checkpoint: {checkpoint_path}")


### 2.3) Refactored

In [30]:
from collections.abc import Sequence
from numbers import Real
from typing import Literal

import numpy as np
import pandas as pd

from simnibs_parcel_analysis.atlas_volume import build_volume_roi_summary
from simnibs_parcel_analysis.utils.prepare_session_file_table import map_session_files

import importlib
import simnibs_parcel_analysis.atlas_volume as atlas_volume

importlib.reload(atlas_volume)
build_volume_roi_summary = atlas_volume.build_volume_roi_summary

def _volume_roi_name(name: str) -> str:
    """Normalize lateralized FreeSurfer labels to the existing lh./rh. ROI format."""
    prefixes = {
        "ctx-lh-": "lh.", "ctx-rh-": "rh.",
        "ctx_lh_": "lh.", "ctx_rh_": "rh.",
        "Left-": "lh.", "Right-": "rh.",
    }
    for prefix, replacement in prefixes.items():
        if name.startswith(prefix):
            return replacement + name[len(prefix):]
    raise ValueError(f"Unsupported or nonlateralized FreeSurfer ROI: {name!r}")

def _prepare_df_efield_volume(
    threshold: int,
    step: str,
    model_type: str,
    percentile: float = 95,
    combine_hemis: bool = False,
    percentile_weighting: Literal["unweighted", "spatial_weighted"] = "spatial_weighted",
    negative_atol: float = 1e-4,
    df_query: pd.DataFrame | None = None,
    asf: str = "asf_5",
    exclude_unmatched: bool = False,
) -> pd.DataFrame:
    """Prepare session-2 courses with aparc+aseg ROI means, percentiles, and CGI outcomes.

    Uses the supplied df_query, or loads fn_query from the existing notebook.
    Also uses fn_cgi, load_course_data, load_stimulation_data, and _compute_ef_sf.
    The query must contain dataset_root and fs_subjects_dir.
    Simulation paths must match df_query.full_file_path exactly.
    With exclude_unmatched=True, omit session-2 courses without a file in the selected query.
    """
    if (isinstance(percentile, (bool, np.bool_)) or not isinstance(percentile, Real)
            or not np.isfinite(percentile) or not 0 <= percentile <= 100):
        raise ValueError("percentile must be a finite number in [0, 100].")
    if not isinstance(combine_hemis, (bool, np.bool_)):
        raise TypeError("combine_hemis must be a boolean.")
    if percentile_weighting not in {"unweighted", "spatial_weighted"}:
        raise ValueError("percentile_weighting must be 'unweighted' or 'spatial_weighted'.")

    # The updated loader supplies dataset_root -> fs_subjects_dir.
    df_query = load_query_results(fn_query) if df_query is None else df_query
    query_columns = ["full_file_path", "subject_dir", "dataset_root", "fs_subjects_dir"]
    missing = set(query_columns) - set(df_query.columns)
    if missing:
        raise ValueError(f"Missing query columns: {sorted(missing)}")
    df_query = df_query.loc[df_query["subject_dir"] != "subj-cat-010-001"].copy()

    query_filters = {
        "model_type_dir": model_type,
        "step": step,
        "asf": asf,
        "threshold": threshold,
    }

    df_cgi = load_course_data(fn_cgi)
    df_stim = _compute_ef_sf(load_stimulation_data(fn_cgi))
    df_stim = df_stim.loc[df_stim["subjid"] != "subj-cat-010-001"].copy()
    df_stim = df_stim.loc[df_stim["person_id"].isin(df_cgi["person_id"])].copy()

    # No fsavg_overlays filter: these are volumetric simulation files.
    mapped = map_session_files(
        sessions=df_stim,
        query=df_query,
        filename="tdcs_uq_gpc.hdf5",
        query_filters=query_filters,
        subject_scan_map="subject_scan_map.json",
        allow_unmatched=exclude_unmatched,
    )

    missing = {"full_file_path", "session_num"} - set(mapped.columns)
    if missing:
        raise ValueError(f"Missing mapped columns: {sorted(missing)}")

    # Preserve the current representative-session definition.
    mapped = mapped.loc[mapped["session_num"] == 2].copy()
    if mapped.empty:
        raise ValueError(f"No eligible session 2 rows for {model_type}, {step}, threshold={threshold}")
    if exclude_unmatched:
        unmatched = mapped["full_file_path"].isna()
        if "mapping_status" in mapped.columns:
            unmatched |= mapped["mapping_status"].ne("matched")
        if unmatched.any():
            detail_columns = [col for col in ("subjid", "series_num", "electrode_placement", "mapping_reason") if col in mapped]
            excluded = mapped.loc[unmatched, detail_columns].drop_duplicates()
            print(f"Excluding {unmatched.sum()} unmatched session-2 course(s) for {model_type}, {step}, threshold={threshold}:")
            print(excluded.to_string(index=False))
            mapped = mapped.loc[~unmatched].copy()
    if mapped.empty:
        raise ValueError(f"No matched session-2 courses for {model_type}, {step}, threshold={threshold}")
    if mapped["full_file_path"].isna().any():
        raise ValueError("Missing simulation file paths in mapped sessions.")

    # Recover metadata by exact simulation path, not clinical subject ID.
    lookup = df_query[query_columns].drop_duplicates().rename(columns={"subject_dir": "modeled_subjid"})
    if lookup.isna().any().any():
        raise ValueError("Missing simulation or FreeSurfer metadata in the query lookup.")
    if lookup["fs_subjects_dir"].astype(str).str.strip().eq("").any():
        raise ValueError("Blank FreeSurfer SUBJECTS_DIR in the query lookup.")
    if lookup["full_file_path"].duplicated().any():
        raise ValueError("A query file path has conflicting subject, dataset, or FreeSurfer metadata.")

    metadata_cols = ["modeled_subjid", "dataset_root", "fs_subjects_dir"]
    temporary = {col: f"_query_{col}" for col in metadata_cols}
    reserved = {*temporary.values(), "_query_match"} & set(mapped.columns)
    if reserved:
        raise ValueError(f"Reserved metadata columns already exist: {sorted(reserved)}")

    mapped = mapped.merge(
        lookup.rename(columns=temporary),
        on="full_file_path",
        how="left",
        sort=False,
        validate="many_to_one",
        indicator="_query_match",
    )

    unmatched = mapped["_query_match"].ne("both")
    if unmatched.any():
        paths = mapped.loc[unmatched, "full_file_path"].unique().tolist()
        raise ValueError(f"Mapped paths do not exactly match df_query.full_file_path: {paths}")

    # Retain matching existing metadata, fill missing metadata, and reject conflicts.
    for col in metadata_cols:
        expected = mapped.pop(temporary[col])
        if col in mapped.columns:
            present = mapped[col].notna()
            if mapped.loc[present, col].ne(expected.loc[present]).any():
                raise ValueError(f"Mapped {col!r} conflicts with metadata from the query inventory.")
        mapped[col] = expected
    mapped = mapped.drop(columns="_query_match")

    # Catch duplicate/missing course identifiers before reading meshes.
    course_keys = ["person_id", "series_num"]
    for label, table in (("Mapped sessions", mapped), ("CGI table", df_cgi)):
        missing = set(course_keys) - set(table.columns)
        if missing:
            raise ValueError(f"{label} is missing course identifiers: {sorted(missing)}")
        if table[course_keys].isna().any().any():
            raise ValueError(f"{label} contains missing course identifiers.")
        if table.duplicated(course_keys).any():
            raise ValueError(f"{label} contains duplicate person_id/series_num combinations.")

    # Repeated courses can reuse one simulation; extract each unique file once.
    inventory = mapped[["full_file_path", *metadata_cols]].drop_duplicates()
    if inventory["full_file_path"].duplicated().any():
        raise ValueError("One simulation file has multiple metadata assignments.")

    roi_frames = []
    for fs_subjects_dir, group in inventory.groupby("fs_subjects_dir", sort=True, observed=True):
        paths = group["full_file_path"].tolist()
        summary = build_volume_roi_summary(
            hdf5_files=paths,
            subjects_dir=fs_subjects_dir,
            subject_ids=group["modeled_subjid"].tolist(),
            simulation_ids=paths,
            atlas_name="aparc",
            field_name="magnE_mean",
            percentile=float(percentile),
            min_elements_per_roi=20,
            freesurfer_lut='/projects/p32903/Alex2/datasets/freesurfer_templates/FreeSurferColorLUT.txt',
            negative_atol=negative_atol,

        )

        if summary.parcel_mean_e is None:
            raise ValueError(f"No ROI means returned for {fs_subjects_dir}")

        means = summary.parcel_mean_e
        percentiles = summary.p95(percentile_weighting)
        if not means.index.equals(percentiles.index) or not means.columns.equals(percentiles.columns):
            raise ValueError("ROI mean and percentile tables do not have matching simulations and ROIs.")
        if means.index.has_duplicates or set(means.index) != set(paths):
            raise ValueError("Extracted simulation IDs do not match the requested unique file paths.")

        means = means.rename(columns=lambda col: f"{_volume_roi_name(col)}_e_mean")
        percentiles = percentiles.rename(columns=lambda col: f"{_volume_roi_name(col)}_e_percentile")
        group_roi = pd.concat([means, percentiles], axis=1)
        group_roi = group_roi.rename_axis("full_file_path").reset_index()

        if not group_roi.columns.is_unique:
            raise ValueError("Duplicate ROI column names after FreeSurfer label normalization.")
        if roi_frames and not group_roi.columns.equals(roi_frames[0].columns):
            raise ValueError(f"ROI columns differ across FreeSurfer datasets: {fs_subjects_dir}")
        roi_frames.append(group_roi)

    df_roi = pd.concat(roi_frames, ignore_index=True)
    if df_roi["full_file_path"].duplicated().any():
        raise ValueError("Duplicate simulation paths in extracted ROI metrics.")

    if combine_hemis:
        lh = {col[3:]: col for col in df_roi.columns if col.startswith("lh.")}
        rh = {col[3:]: col for col in df_roi.columns if col.startswith("rh.")}
        if not lh or set(lh) != set(rh):
            raise ValueError(f"Unmatched hemisphere ROIs: {sorted(set(lh) ^ set(rh))}")

        # Average matching summaries; this is not a pooled bilateral percentile.
        bilateral = pd.DataFrame({
            roi: (df_roi[lh[roi]] + df_roi[rh[roi]]) / 2
            for roi in sorted(lh)
        }, index=df_roi.index)
        df_roi = pd.concat([df_roi[["full_file_path"]], bilateral], axis=1)

    roi_columns = df_roi.columns.drop("full_file_path").tolist()
    for label, table in (("mapped sessions", mapped), ("CGI table", df_cgi)):
        overlap = set(roi_columns) & set(table.columns)
        if overlap:
            raise ValueError(f"ROI columns conflict with {label}: {sorted(overlap)}")

    df_stim_roi = mapped.merge(df_roi, on="full_file_path", how="left", sort=False, validate="many_to_one")
    if df_stim_roi[roi_columns].isna().any().any():
        raise ValueError("Missing ROI metrics after merging simulation files.")

    shared_metadata = [col for col in ("age", "sex", "dob") if col in df_stim_roi and col in df_cgi]
    return df_stim_roi.drop(columns=shared_metadata).merge(
        df_cgi,
        on=course_keys,
        how="left",
        sort=False,
        validate="one_to_one",
        suffixes=("_stim", "_cgi"),
    )


"""Run these functions in the existing notebook with the clinical loaders and
run_roi_correlations defined. The atlas remains fixed to aparc+aseg.
"""

from datetime import datetime
from hashlib import sha256
from itertools import product
from numbers import Real
from pathlib import Path
import time

import numpy as np
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

def _save_correlation_workbook(tables: dict[str, pd.DataFrame], output: Path) -> None:
    """Write tables with filters, frozen headers, and display-only numeric formatting."""
    temporary = output.with_name(f"{output.stem}.tmp.xlsx")
    with pd.ExcelWriter(temporary, engine="openpyxl") as writer:
        for name, table in tables.items():
            table.to_excel(writer, sheet_name=name, index=False, na_rep="")
            sheet = writer.sheets[name]
            sheet.freeze_panes = "A2"
            sheet.auto_filter.ref = sheet.dimensions
            sheet.row_dimensions[1].height = 30

            for cell in sheet[1]:
                cell.font = Font(bold=True, color="FFFFFF")
                cell.fill = PatternFill("solid", fgColor="234E70")
                cell.alignment = Alignment(wrap_text=True, vertical="center")

            for j, column in enumerate(table.columns, start=1):
                width = min(36, max(14, len(column) + 2))
                if column in {"roi", "roi_column", "x", "config_label"}:
                    width = 38
                sheet.column_dimensions[get_column_letter(j)].width = width
                number_format = "0.000E+00" if column in {"p", "p_bh_sweep"} else "0.000"
                if column in {"r", "r_abs", "p", "p_bh_sweep", "prepare_seconds"}:
                    for cells in sheet.iter_rows(min_row=2, min_col=j, max_col=j):
                        cells[0].number_format = number_format

            if name == "settings":
                sheet.column_dimensions["A"].width = 30
                sheet.column_dimensions["B"].width = 60
                for cells in sheet.iter_rows(min_row=2):
                    cells[1].alignment = Alignment(wrap_text=True, vertical="top")
                    sheet.row_dimensions[cells[0].row].height = 45

    temporary.replace(output)

def run_volume_roi_correlations(
    df_query: pd.DataFrame | None = None,
    *,
    combine_hemis: bool | Sequence[bool] = False,
    suffix: str = "",
    percentiles: Sequence[float] = (95,),
    percentile_weightings: Sequence[str] = ("spatial_weighted",),
    methods: Sequence[str] = ("spearman",),
    steps: Sequence[str] = ("step_0", "step_4"),
    asf: str = "asf_5",
    exclude_unmatched: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run the original aparc+aseg ROI correlation sweep, writing CSV/Excel/checkpoint.

    A supplied df_query is used for BOTH configuration discovery and ROI preparation.
    combine_hemis accepts one bool or a sequence of unique bools (e.g. (False, True)).
    suffix adds a directory under results/correlations/aparc/.
    exclude_unmatched=True excludes session-2 courses without files in df_query,
    and prints an exclusion report; the strict behavior is retained by default.
    """
    atlas_name = "aparc"
    if not isinstance(suffix, str) or (suffix and (Path(suffix).name != suffix or suffix in {".", ".."})):
        raise ValueError("suffix must be a single folder name (or an empty string).")

    combine_hemis_options = ([combine_hemis] if isinstance(combine_hemis, (bool, np.bool_))
                             else list(combine_hemis))
    supplied_query = df_query is not None
    df_query = load_query_results(fn_query) if df_query is None else df_query

    output_dir = Path("results/correlations") / atlas_name / suffix
    output_dir.mkdir(parents=True, exist_ok=True)
    xlsx_path = output_dir / "roi_efield.xlsx"
    csv_path = output_dir / "roi_efield.csv"
    checkpoint_path = output_dir / "roi_efield_checkpoint.csv"

    # Avoid mixing a new run with an older run or partial checkpoint.
    for path in (xlsx_path, csv_path, checkpoint_path):
        if path.exists():
            raise FileExistsError(f"Output already exists; use a fresh folder or move the previous run: {path}")

    for name, values in (
        ("percentiles", percentiles), ("percentile_weightings", percentile_weightings),
        ("methods", methods), ("combine_hemis_options", combine_hemis_options),
    ):
        if not values or len(set(values)) != len(values):
            raise ValueError(f"{name} must be nonempty and contain no duplicates.")
    if any(isinstance(p, (bool, np.bool_)) or not isinstance(p, Real)
           or not np.isfinite(p) or not 0 <= p <= 100 for p in percentiles):
        raise ValueError("percentiles must contain finite numbers in [0, 100].")
    if not set(percentile_weightings) <= {"unweighted", "spatial_weighted"}:
        raise ValueError("Unsupported percentile weighting.")
    if not set(methods) <= {"pearson", "spearman"}:
        raise ValueError("Unsupported correlation method.")
    if not all(isinstance(value, (bool, np.bool_)) for value in combine_hemis_options):
        raise TypeError("combine_hemis_options must contain booleans.")

    # Select all simulation configurations actually present in the supplied query.
    config_columns = ["model_type_dir", "step", "threshold"]
    required = {*config_columns, "asf", "subject_dir", "full_file_path"}
    missing = required - set(df_query.columns)
    if missing:
        raise ValueError(f"Missing query columns: {sorted(missing)}")

    available = df_query.loc[
        df_query["asf"].eq(asf)
        & df_query["subject_dir"].ne("subj-cat-010-001")
        & df_query["full_file_path"].str.endswith("/tdcs_uq_gpc.hdf5", na=False)
    ].copy()

    available = available.loc[available["step"].isin(steps)].copy()

    if available.empty or available[config_columns].isna().any().any():
        raise ValueError("No eligible HDF5 configurations, or missing simulation configuration values.")
    threshold_values = pd.to_numeric(available["threshold"], errors="raise")
    if (available["threshold"].map(lambda value: isinstance(value, (bool, np.bool_))).any()
            or not np.isfinite(threshold_values).all() or threshold_values.mod(1).ne(0).any()):
        raise ValueError("Simulation thresholds must be finite integers.")
    available["threshold"] = threshold_values.astype(int)

    base_configs = list(
        available[config_columns].drop_duplicates().sort_values(config_columns).itertuples(index=False, name=None)
    )
    configs = [
        (*base, percentile, combine_hemis, weighting)
        for base, percentile, combine_hemis, weighting in product(
            base_configs, percentiles, combine_hemis_options, percentile_weightings
        )
    ]

    # Means run once per simulation/hemisphere/method, not once per percentile/weighting.
    total_batches = (len(base_configs) * len(combine_hemis_options) * len(methods)
                     * (1 + len(percentiles) * len(percentile_weightings)))
    results, analysis_records = [], []
    started_at = datetime.now().astimezone().isoformat(timespec="seconds")

    print(f"Simulation configurations: {len(base_configs)}")
    print(f"Preparation calls: {len(configs)}")
    print(f"Correlation batches: {total_batches}")

    with tqdm(total=total_batches, desc="aparc ROI correlations", unit="batch", mininterval=0, miniters=1) as pbar:
        for model_type, step, threshold, percentile, combine_hemis, weighting in configs:
            progress = {
                "model": model_type, "step": step, "threshold": threshold,
                "percentile": percentile, "weighting": weighting, "hemis": combine_hemis,
            }
            pbar.set_postfix(**progress, status="preparing", refresh=True)
            start = time.perf_counter()

            df = _prepare_df_efield_volume(
                threshold=int(threshold),
                step=step,
                model_type=model_type,
                percentile=percentile,
                combine_hemis=combine_hemis,
                percentile_weighting=weighting,
                df_query=df_query,
                asf=asf,
                exclude_unmatched=exclude_unmatched,
            )
            prepare_seconds = time.perf_counter() - start

            # This identifies the PREPARED course set; n_courses still reports usable pairs per ROI.
            course_keys = df[["person_id", "series_num"]].sort_values(["person_id", "series_num"])
            prepared_cohort_id = sha256(course_keys.to_csv(index=False).encode()).hexdigest()[:16]
            dataset_roots = ";".join(sorted(df["dataset_root"].unique()))
            first_summary = percentile == percentiles[0] and weighting == percentile_weightings[0]
            metrics = ["mean", "percentile"] if first_summary else ["percentile"]

            for metric, method in product(metrics, methods):
                pbar.set_postfix(**progress, metric=metric, method=method, status="correlating", refresh=True)
                corr = run_roi_correlations(df, combine_hemis=combine_hemis, metric=metric, method=method).copy()
                if corr.empty:
                    raise ValueError(f"No ROI correlation results for {progress}, metric={metric}, method={method}")

                metric_label = f"p{float(percentile):g}_{weighting}" if metric == "percentile" else "mean"
                hemis_label = "bilateral" if combine_hemis else "separate"
                config_label = f"{model_type}|{step}|thr{threshold}|{metric_label}|{hemis_label}|{method}"
                metadata = {
                    "analysis_id": len(analysis_records) + 1,
                    "config_label": config_label,
                    "atlas": atlas_name,
                    "atlas_file": "aparc+aseg.mgz",
                    "asf": asf,
                    "model_type": model_type,
                    "step": step,
                    "threshold": int(threshold),
                    "metric": metric,
                    "percentile": float(percentile) if metric == "percentile" else np.nan,
                    "percentile_weighting": weighting if metric == "percentile" else pd.NA,
                    "mean_weighting": "unweighted_elements" if metric == "mean" else pd.NA,
                    "combine_hemis": bool(combine_hemis),
                    "hemisphere_aggregation": "mean_of_hemisphere_summaries" if combine_hemis else "none",
                    "method": method,
                    "field_name": "magnE_mean",
                    "representative_session": 2,
                    "dataset_roots": dataset_roots,
                    "prepared_cohort_id": prepared_cohort_id,
                }
                corr = corr.assign(**metadata, r_abs=corr["r"].abs())

                # Persist completed batches before the expensive next preparation call.
                corr.to_csv(checkpoint_path, mode="a", header=not checkpoint_path.exists(), index=False)
                results.append(corr)
                analysis_records.append({
                    **metadata,
                    "n_roi_results": len(corr),
                    "n_ok": int(corr["status"].eq("ok").sum()),
                    "n_courses_min": int(corr["n_courses"].min()),
                    "n_courses_max": int(corr["n_courses"].max()),
                    "n_patients_min": int(corr["n_patients"].min()),
                    "n_patients_max": int(corr["n_patients"].max()),
                    "prepare_seconds": prepare_seconds,
                })
                pbar.update(1)
                pbar.refresh()

            pbar.set_postfix(**progress, status="done", last_sec=round(time.perf_counter() - start, 1), refresh=True)

    if len(analysis_records) != total_batches:
        raise RuntimeError("Completed correlation batch count does not match the planned count.")

    df_results = pd.concat(results, ignore_index=True)
    df_analyses = pd.DataFrame(analysis_records)
    test_keys = [
        "atlas", "asf", "model_type", "step", "threshold", "metric", "percentile",
        "percentile_weighting", "combine_hemis", "method", "hemisphere", "roi", "y",
    ]
    if df_results.duplicated(test_keys).any():
        raise ValueError("Duplicate ROI tests in the final results; do not silently deduplicate them.")

    # Exploratory BH adjustment across ALL valid tests in this completed sweep.
    # This does not correct invalid/unclustered input p-values.
    df_results["p_bh_sweep"] = np.nan
    valid = df_results["status"].eq("ok")
    p_values = pd.to_numeric(df_results.loc[valid, "p"], errors="raise")
    if not np.isfinite(p_values).all() or not p_values.between(0, 1).all():
        raise ValueError("An 'ok' correlation has an invalid p-value.")
    if valid.any():
        df_results.loc[valid, "p_bh_sweep"] = multipletests(p_values, method="fdr_bh")[1]

    first_columns = [
        "analysis_id", "config_label", "atlas", "model_type", "step", "threshold", "metric",
        "percentile", "percentile_weighting", "mean_weighting", "combine_hemis", "hemisphere", "roi",
        "method", "r", "r_abs", "p", "p_bh_sweep", "n_courses", "n_patients", "n_missing", "status",
    ]
    df_results = df_results[first_columns + [c for c in df_results.columns if c not in first_columns]]

    settings = {
        "started_at": started_at,
        "completed_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "query_file": str(Path(fn_query).resolve()),
        "query_selection": "supplied df_query" if supplied_query else "loaded query file",
        "steps": str(list(steps)),
        "output_suffix": suffix,
        "exclude_unmatched_session2": str(exclude_unmatched),
        "clinical_file": str(Path(fn_cgi).resolve()),
        "atlas": atlas_name,
        "atlas_file": "aparc+aseg.mgz",
        "simulation_selection": f"Existing HDF5 configurations in the query for {asf}, excluding subj-cat-010-001",
        "percentiles": str(list(percentiles)),
        "percentile_weightings": str(list(percentile_weightings)),
        "methods": str(list(methods)),
        "combine_hemis_options": str(combine_hemis_options),
        "representative_session": "2; unchanged from the preparation function",
        "min_elements_per_roi": "20; unchanged from the preparation function",
        "outcome": "cgi_change as supplied by the clinical loader; not recomputed",
        "mean_definition": "Unweighted arithmetic mean across tetrahedral elements; analyzed only once per configuration",
        "percentile_definitions": "unweighted: equal elements; spatial_weighted: tetrahedron-volume-weighted empirical CDF",
        "bilateral_definition": "Arithmetic mean of left/right summary values, not a pooled bilateral percentile",
        "mean_parameter_blanks": "percentile and percentile_weighting are blank because neither defines the ROI mean",
        "prepared_cohort_id": "Hash of sorted prepared person_id/series_num pairs; not a hash of all ROI-specific usable pairs",
        "multiplicity": "p_bh_sweep: BH across all finite status=ok tests in this completed atlas sweep",
        "p_value_limitation": "Existing two-sided Pearson/Spearman tests are unchanged; no patient-cluster correction",
        "checkpoint": "Completed batches only; no full-sweep BH adjustment and no automatic resume",
    }
    df_settings = pd.DataFrame(settings.items(), columns=["setting", "value"])

    df_results.to_csv(csv_path, index=False)
    _save_correlation_workbook({"results": df_results, "analyses": df_analyses, "settings": df_settings}, xlsx_path)

    print(f"Saved {len(df_results)} ROI results from {len(df_analyses)} analysis batches.")
    print(f"Excel: {xlsx_path}")
    print(f"CSV: {csv_path}")
    print(f"Checkpoint: {checkpoint_path}")

    return df_results, df_analyses


In [37]:
df_query = load_query_results(fn_query)
df_query = df_query[~df_query['electrode_configuration'].isin(['RUL', 'BF'])].copy()
df_subset = df_query[df_query['threshold']!=80].copy()

df_results, df_analyses = run_volume_roi_correlations(
    df_query=df_subset,
    combine_hemis=False,
    suffix="bilateral_new",
    percentiles=(95,),
    percentile_weightings=("spatial_weighted",),
    exclude_unmatched=True,
    methods=("spearman",),
)

Simulation configurations: 8
Preparation calls: 8
Correlation batches: 16


aparc ROI correlations:   0%|          | 0/16 [00:00<?, ?batch/s, hemis=0, model=skin_double, percentile=95, status=preparing, step=step_0, threshold=90, weighting=spatial_weighted]

Excluding 22 unmatched session-2 course(s) for skin_double, step_0, threshold=90:
          subjid  series_num electrode_placement
subj-cat-002-001           1                  bt
subj-cat-005-001           1                  bt
subj-cat-012-001           1                  bl
subj-cat-019-001           1                  bl
subj-cat-025-001           2                  bl
subj-cat-027-001           1                  bl
subj-cat-028-001           1                  bl
subj-cat-046-001           1                  bl
subj-cat-051-001           1                  bl
subj-cat-057-001           1                  bl
subj-cat-058-001           1                  bl
subj-cat-060-003           1                  bl
subj-cat-060-003           2                  bl
subj-cat-067-001           1                  bt
subj-cat-022-001           1                  bl
subj-cat-022-001           2                  bl
subj-cat-022-001           3                  bl
subj-cat-033-001           1        

FileNotFoundError: HDF5 files do not exist:
/gpfs/projects/b32903/Alex/results/asf_5/STU00225089/skin_double/subj-cat-019-001/RUL/90/20261007_102014/step_0/tdcs_uq_gpc.hdf5